In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_train_df.parquet")
print(df.columns)


# Data Loading

We load the pre-processed training (2015 - 2018) and blind test (2019) datasets. These datasets have finished our feature engineering pipeline.

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
from pyspark.sql.functions import regexp_replace, mean, stddev, log1p, concat_ws
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

spark.conf.set("spark.sql.ansi.enabled", "false")

TEAM_DIR = "dbfs:/student-groups/Group_3_2"
otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_test_df.parquet")

from pyspark.sql.functions import hour

# Extract hour of day from scheduled departure timestamp (matches FE notebook)
otpw_train_df = otpw_train_df.withColumn("DEP_HOUR", hour("sched_depart_date_time"))
otpw_test_df = otpw_test_df.withColumn("DEP_HOUR", hour("sched_depart_date_time"))

# Cast DISTANCE to numeric (stored as string in parquet)
otpw_train_df = otpw_train_df.withColumn("DISTANCE", col("DISTANCE").cast("double"))
otpw_test_df = otpw_test_df.withColumn("DISTANCE", col("DISTANCE").cast("double"))

print(f"Train (2015 - 2018): {otpw_train_df.count():,} rows x {len(otpw_train_df.columns)} columns")
print(f"Test  (2019):      {otpw_test_df.count():,} rows x {len(otpw_test_df.columns)} columns")

### Checkpointing

Training on 4 years of flight data is expensive, a single GBT grid search takes over 2 hours, also when the Databricks cluster auto-terminates after 30 minutes of idle time, we lose all computed results.

To avoid re-running completed experiments, we save CV results, blind test results, grid search outputs, and threshold tuning tables as parquet files on DBFS. Each file is prefixed with a date stamp so results from different feature engineering iterations can coexist. The pattern is: if the checkpoint exists, load it and skip; otherwise, compute and save.

In [0]:
import os

# pandas requires /dbfs/ local path, not dbfs: URI
TEAM_DIR_LOCAL = TEAM_DIR.replace("dbfs:", "/dbfs")
CHECKPOINT_DATE = "041226"

def save_checkpoint(df, name):
    path = f"{TEAM_DIR_LOCAL}/{CHECKPOINT_DATE}_{name}.parquet"
    df.to_parquet(path)
    print(f"Checkpoint saved: {CHECKPOINT_DATE}_{name}")

def load_checkpoint(name):
    path = f"{TEAM_DIR_LOCAL}/{CHECKPOINT_DATE}_{name}.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"Checkpoint loaded: {CHECKPOINT_DATE}_{name} ({len(df)} rows)")
        return df
    return None

In [0]:
label_col = "DEP_DEL15"

# Verify class balance
train_dist = otpw_train_df.groupBy(label_col).count().toPandas()
train_dist["pct"] = (train_dist["count"] / train_dist["count"].sum() * 100).round(2)
print(f"\nTrain class distribution:\n{train_dist}")

Modeling Pipeline

We apply the modeling pipeline to the 4-year training dataset (2015 - 2018) with engineered features, evaluating Logistic Regression, Random Forest, Gradient Boosted Trees, and a Multilayer Perceptron. The 2019 dataset is held out as a blind test set that is never consulted during training or cross-validation.

### Build Spark ML Pipeline

The pipeline applies `StringIndexer`, `OneHotEncoder` for each categorical feature, then assembles all numeric and encoded categorical features into a single vector using `VectorAssembler`.

In [0]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Shared categorical features (used by LR and tree-based models)
categorical_features = [
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR",
    "VisibilityCat",
]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

# 1. Logistic Regression Pipeline (Baseline)
#    Replaces delayed_flights_2_4hr_before with log_total_flights_2_4hr_before
#    to avoid multicollinearity (count = pct x total)
lr_numeric_features = [
    # Weather
    "2h_HourlyDryBulbTemperature_stdized",
    "2h_HourlyWindSpeed_log_stdized",
    
    "PrecipBinary",
    "2h_HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (log of total flights replaces raw count)
    "delay_pct_flights_2_4hr_before", "log_total_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
]

lr_assembler_inputs = lr_numeric_features + [f"{c}_ohe" for c in categorical_features]
lr_assembler = VectorAssembler(inputCols=lr_assembler_inputs, outputCol="features", handleInvalid="skip")
lr_pipeline_stages = indexers + encoders + [lr_assembler]

# 2. Tree-based Pipeline (RF, GBT)
tree_numeric_features = [
    # Weather
    "2h_HourlyDryBulbTemperature_stdized",
    "2h_HourlyWindSpeed_log_stdized",
    "PrecipBinary",
    "2h_HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (short-term airport congestion)
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
]

tree_assembler_inputs = tree_numeric_features + [f"{c}_ohe" for c in categorical_features]
tree_assembler = VectorAssembler(inputCols=tree_assembler_inputs, outputCol="features", handleInvalid="skip")
tree_pipeline_stages = indexers + encoders + [tree_assembler]

# 3. MLP Pipeline (reduced feature set for neural network)
#    Drops ORIGIN OHE (high cardinality might high dimentional sparse inputs for MLP), uses PageRank instead
mlp_numeric_features = [
    # Weather
    "2h_HourlyDryBulbTemperature_stdized", "2h_HourlyWindSpeed_log_stdized",
    "PrecipBinary", "2h_HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (replaces ORIGIN/DEST)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
]

mlp_categorical_features = [
    "OP_UNIQUE_CARRIER", "DAY_OF_WEEK", "MONTH", "DEP_HOUR", "VisibilityCat",
]

mlp_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_mlp_idx", handleInvalid="keep")
    for c in mlp_categorical_features
]
mlp_encoders = [
    OneHotEncoder(inputCol=f"{c}_mlp_idx", outputCol=f"{c}_mlp_ohe")
    for c in mlp_categorical_features
]
mlp_assembler_inputs = mlp_numeric_features + [f"{c}_mlp_ohe" for c in mlp_categorical_features]
mlp_assembler = VectorAssembler(inputCols=mlp_assembler_inputs, outputCol="features", handleInvalid="skip")
mlp_pipeline_stages = mlp_indexers + mlp_encoders + [mlp_assembler]

# Determine MLP input dimension from training data
mlp_prep = Pipeline(stages=mlp_pipeline_stages).fit(otpw_train_df)
sample_vec = mlp_prep.transform(otpw_train_df.limit(1)).select("features").head()[0]
mlp_input_dim = len(sample_vec)

# Summary
print(f"LR:    {len(lr_numeric_features)} numeric + {len(categorical_features)} categorical = {len(lr_assembler_inputs)} assembler inputs")
print(f"Tree:  {len(tree_numeric_features)} numeric + {len(categorical_features)} categorical = {len(tree_assembler_inputs)} assembler inputs")
print(f"MLP:   {len(mlp_numeric_features)} numeric + {len(mlp_categorical_features)} categorical = {mlp_input_dim} input dim (after OHE)")


### Evaluation Metrics

For airlines, the cost of a **false negative** (predicting on-time when the flight is actually delayed) is significantly higher than a **false positive** (predicting delayed when the flight departs on time):

- **False Negative (missed delay):** The airline fails to prepare: crew scheduling is disrupted last-minute, gate reassignments increases, passengers miss connections, and a single missed delay can trigger a chain of downstream issues. The financial and reputational cost is high.
- **False Positive (false alarm):** The airline pre-allocates standby resources that weren't needed: extra crew on standby, a backup gate reserved, passengers notified of a potential delay. The cost is moderate (some wasted resources) but overall harmless.

This motivates our choice of **F2-score** as the primary evaluation metric. F2 combines precision and recall into a single score but places twice the weight on recall, penalizing missed delays more heavily than false alarms. We use F2 rather than raw recall because optimizing recall alone is trivial (predict every flight as delayed), while F2 ensures the model maintains reasonable precision.

| Metric  | Role | Why |
|--------|------|-----|
| **F2**  | Primary | Balances precision and recall with 2x weight on recall, preventing missed delays while keeping predictions actionable |
| **Recall** | Supporting | Measures raw delay detection rate; F2's recall-weighted design ensures this stays high |
| **Precision** | Supporting | Ensures predicted delays are actionable, not just noise |
| **AUC-PR**  | Supporting | Captures the full precision-recall trade-off across all thresholds |

In [0]:
def evaluate_model(predictions, label_col="DEP_DEL15"):
    evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")

    tp = predictions.filter((col("prediction") == 1) & (col(label_col) == 1)).count()
    fp = predictions.filter((col("prediction") == 1) & (col(label_col) == 0)).count()
    fn = predictions.filter((col("prediction") == 0) & (col(label_col) == 1)).count()

    precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f2 = round(5 * precision * recall / (4 * precision + recall), 4) if (precision + recall) > 0 else 0.0

    return {
        "AUC-PR": round(evaluator_pr.evaluate(predictions), 4),
        "Precision": precision,
        "Recall": recall,
        "F2": f2,
    }

### Modeling Approaches

We use **Logistic Regression** as the baseline. We then compare against tree-based models and a neural network:

- **Random Forest:** Handles non-linear feature interactions without explicit engineering. Robust to outliers and noisy features.
- **Gradient Boosted Trees:** Sequentially builds trees to correct previous errors, typically achieving higher predictive performance than Random Forest. More prone to overfitting.
- **Multilayer Perceptron (MLP):** A feed-forward neural network that learns non-linear decision boundaries through hidden layers. We use a reduced feature set for MLP that replaces high-cardinality categorical features (`ORIGIN`) with continuous graph-based representations (PageRank) to avoid the dimensionality issues caused by one-hot encoding in dense neural networks.

In [0]:
def run_experiment(model, pipeline_stages, train_df, test_df, model_name, dataset_name):
    full_pipeline = Pipeline(stages=pipeline_stages + [model])

    start = time.time()
    fitted = full_pipeline.fit(train_df)
    train_time = round(time.time() - start, 1)

    train_preds = fitted.transform(train_df)
    test_preds = fitted.transform(test_df)

    train_metrics = evaluate_model(train_preds)
    test_metrics = evaluate_model(test_preds)

    print(f"\n============================================================")
    print(f"{model_name} on {dataset_name}")
    print(f"Training time: {train_time}s")
    print(f"\n============================================================")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"------------------------------------------")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Train Time (s)": train_time,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }, fitted

### Class Imbalance Handling

The training data exhibits a natural class imbalance of approximately 80% on-time flights and 20% delayed flights. Without correction, models trained on this distribution would be biased toward predicting the majority class (on-time), resulting in high accuracy but poor recall for delays.

We apply **random undersampling** to the majority class (on-time flights) to match the minority class count, creating a 50/50 balanced training set. This forces models to learn delay patterns rather than defaulting to the majority class. Undersampling is applied **only to training data**, and all test and validation sets retain their natural class distribution to accurately simulate real-world conditions. During cross-validation, undersampling is performed independently within each fold after the temporal split.

In [0]:
def undersample_majority(df, label_col="DEP_DEL15", seed=42):
    """Undersample the majority class (label=0) to match the minority class count."""
    major_df = df.filter(col(label_col) == 0)
    minor_df = df.filter(col(label_col) == 1)
    major_count = major_df.count()
    minor_count = minor_df.count()
    undersample_ratio = minor_count / major_count
    major_undersampled = major_df.sample(withReplacement=False, fraction=undersample_ratio, seed=seed)
    return major_undersampled.unionAll(minor_df)

## Cross-Validation (2015 - 2018)

We use **yearly single-year cross-validation** on the 4-year training set:

| Fold | Train | Validate |
|------|-------|----------|
| 1 | 2015 | 2016 |
| 2 | 2016 | 2017 |
| 3 | 2017 | 2018 |

Each fold trains on one year and validates on the next, providing three independent temporal evaluations. After cross-validation, we train on the full 2015 - 2018 dataset and evaluate on the held-out 2019 test set.

### Leakage Prevention

To ensure no future information leaks into training, we enforce the following safeguards in every fold:

1. **Temporal separation:** Each fold trains only on a past year and validates on the next year. No random shuffling across time.
2. **Undersampling on train only:** Majority-class downsampling is applied only to the training fold. Validation and test data retain their natural class distribution to accurately simulate real-world conditions.
3. **No target leakage:** We predict `DEP_DEL15` (binary delay indicator) using only features available before departure: weather observations, schedule metadata, and airport/carrier identifiers. No post-departure columns (actual delay minutes, arrival info) are included.

### Overfitting Detection

We report **both train and validation F2** for every fold. A large gap (train >> validation) signals overfitting, the model memorizes training data but fails to generalize. A smaller gap indicates better generalization.

In [0]:
def rolling_cv(df, model_fn, pipeline_stages, folds, label_col="DEP_DEL15"):
    """Rolling cross-validation with yearly folds and majority undersampling.
    Reports both train and test metrics per fold for overfitting analysis."""
    fold_results = []
    fitted_models = []

    for i, fold in enumerate(folds, 1):
        train_fold = df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        # Undersample majority class in the training fold
        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        model = model_fn()
        full_pipeline = Pipeline(stages=pipeline_stages + [model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        fitted_models.append(fitted)

        # Evaluate on both train and test
        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    fold_df = pd.DataFrame(fold_results)
    return fold_df, fitted_models

In [0]:
cv_folds = [
    {"train_start": 2015, "train_end": 2015, "test_start": 2016, "test_end": 2016, "label": "Train 2015, Val 2016"},
    {"train_start": 2016, "train_end": 2016, "test_start": 2017, "test_end": 2017, "label": "Train 2016, Val 2017"},
    {"train_start": 2017, "train_end": 2017, "test_start": 2018, "test_end": 2018, "label": "Train 2017, Val 2018"},
]

## Model Comparison with Default Hyperparameters

We run rolling CV on all four models using default hyperparameters to establish baseline performance. Each model is trained on 3 yearly folds with undersampling, reporting train and validation metrics per fold to assess both performance and overfitting.

### Logistic Regression (Baseline)

In [0]:
lr_cv = load_checkpoint("cv3_logistic_regression")
if lr_cv is None:
    lr_cv, lr_cv_models = rolling_cv(
        otpw_train_df,
        model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
        pipeline_stages=lr_pipeline_stages, folds=cv_folds
    )
    save_checkpoint(lr_cv, "cv3_logistic_regression")
else:
    lr_cv_models = []

### Random Forest

In [0]:
rf_cv = load_checkpoint("cv3_random_forest")
if rf_cv is None:
    rf_cv, rf_cv_models = rolling_cv(
        otpw_train_df,
        model_fn=lambda: RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
        pipeline_stages=tree_pipeline_stages, folds=cv_folds
    )
    save_checkpoint(rf_cv, "cv3_random_forest")
else:
    rf_cv_models = []

### Gradient Boosted Trees

In [0]:
gbt_cv = load_checkpoint("cv3_gradient_boosted_trees")
if gbt_cv is None:
    gbt_cv, gbt_cv_models = rolling_cv(
        otpw_train_df,
        model_fn=lambda: GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
        pipeline_stages=tree_pipeline_stages, folds=cv_folds
    )
    save_checkpoint(gbt_cv, "cv3_gradient_boosted_trees")
else:
    gbt_cv_models = []

### MLP (1 Hidden Layer)

Feed forward neural network with one hidden layer (`[input_dim, 64, 2]`, `maxIter=100`). Uses `mlp_pipeline_stages` which drops `ORIGIN` one-hot encoding (300+ airports will cause huge sparse vector) and relies on continuous PageRank features instead, because dense neural networks perform poorly with high dimensional sparse inputs.

In [0]:
mlp_cv = load_checkpoint("cv3_mlp_shallow")
if mlp_cv is None:
    # Custom CV loop: MLP input dimension varies per fold because each temporal
    # split has different categorical cardinalities after OHE.
    fold_results = []
    mlp_cv_models = []

    for i, fold in enumerate(cv_folds, 1):
        train_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        # Compute actual feature dimension for this fold's training data
        prep_pipeline = Pipeline(stages=mlp_pipeline_stages).fit(train_balanced)
        sample_vec = prep_pipeline.transform(train_balanced).select("features").first()[0]

        fold_input_dim = len(sample_vec)

        mlp_model = MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[fold_input_dim, 64, 2], maxIter=100, blockSize=128, seed=42
        )
        full_pipeline = Pipeline(stages=mlp_pipeline_stages + [mlp_model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        mlp_cv_models.append(fitted)

        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    mlp_cv = pd.DataFrame(fold_results)
    save_checkpoint(mlp_cv, "cv3_mlp_shallow")
else:
    mlp_cv_models = []

### MLP (2 Hidden Layers)

A deeper MLP architecture with two hidden layers (`[input_dim, 128, 64, 2]`) to test whether additional depth improves delay prediction. The deeper network can learn hierarchical feature representations but is more likely to overfit and slower to train.

In [0]:
mlp_deep_cv = load_checkpoint("cv3_mlp_deep")
if mlp_deep_cv is None:
    fold_results = []
    mlp_deep_cv_models = []

    for i, fold in enumerate(cv_folds, 1):
        train_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        # Compute feature dimension for this fold
        prep_pipeline = Pipeline(stages=mlp_pipeline_stages).fit(train_balanced)
        sample_vec = prep_pipeline.transform(train_balanced).select("features").first()[0]
        fold_input_dim = len(sample_vec)

        # Deep MLP: 2 hidden layers [input, 128, 64, 2]
        mlp_model = MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[fold_input_dim, 128, 64, 2], maxIter=100, blockSize=128, seed=42
        )
        full_pipeline = Pipeline(stages=mlp_pipeline_stages + [mlp_model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        mlp_deep_cv_models.append(fitted)

        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    mlp_deep_cv = pd.DataFrame(fold_results)
    save_checkpoint(mlp_deep_cv, "cv3_mlp_deep")
else:
    mlp_deep_cv_models = []

### LSTM (Extra Credit)

Flight delays has **sequential dependencies** that feedforward models treat as independent events:

1. **Aircraft propagation:** The same aircraft (`TAIL_NUM`) flies multiple legs per day. A delay on one leg cascades to the next, for example, if an aircraft arrives 30 minutes late at LAX, its next departure from LAX is likely delayed. LSTM's hidden state carries this "delay memory" across legs.
2. **Airport congestion buildup:** Delays at a busy airport accumulate over hours. LSTM could learn this better.

Unlike feedforward networks (LR, GBT, MLP) which treat each flight independently, LSTM explicitly models temporal ordering.

In [0]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score

# --- LSTM constants ---
SEQ_LEN = 5
LSTM_EPOCHS = 10
lstm_input_features = mlp_numeric_features  # same numeric features as MLP (no target)
lstm_columns_to_collect = ["TAIL_NUM", "sched_depart_date_time", "YEAR"] + lstm_input_features + ["DEP_DEL15"]

# --- LSTM model definition ---
class FlightLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]  # Take last timestep
        out = self.dropout(last_hidden)
        return self.fc(out).squeeze(-1)

def build_sequences(df, seq_len, feature_cols, label_col):
    """Build fixed-length sequences per TAIL_NUM. Pad with zeros if fewer than seq_len previous flights."""
    X_seqs, y_labels = [], []
    for tail, group in df.groupby("TAIL_NUM"):
        features = group[feature_cols].fillna(0).values.astype(np.float32)
        labels = group[label_col].fillna(0).values.astype(np.float32)
        for i in range(len(group)):
            start = max(0, i - seq_len)
            seq = features[start:i+1]
            if len(seq) < seq_len + 1:
                pad = np.zeros((seq_len + 1 - len(seq), len(feature_cols)), dtype=np.float32)
                seq = np.vstack([pad, seq])
            X_seqs.append(seq)
            y_labels.append(labels[i])
    return np.array(X_seqs), np.array(y_labels)

def evaluate_numpy(y_true, y_pred, y_probs):
    """Evaluate binary predictions on numpy arrays. Returns same dict format as evaluate_model."""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    precision = round(float(tp / (tp + fp)), 4) if (tp + fp) > 0 else 0.0
    recall = round(float(tp / (tp + fn)), 4) if (tp + fn) > 0 else 0.0
    f2 = round(5 * precision * recall / (4 * precision + recall), 4) if (precision + recall) > 0 else 0.0
    auc_pr = round(float(average_precision_score(y_true, y_probs)), 4)

    return {"AUC-PR": auc_pr, "Precision": precision, "Recall": recall, "F2": f2}

def train_lstm(X_train, y_train, input_dim, epochs=10, hidden_dim=64,
               num_layers=2, dropout=0.3, lr=0.001, batch_size=4096):
    """Train a FlightLSTM model. Returns (model, device, train_time)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FlightLSTM(input_dim, hidden_dim, num_layers, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(loader):.4f}")

    train_time = round(time.time() - start, 1)
    return model, device, train_time

def predict_lstm(model, X, device, batch_size=4096):
    """Get predicted probabilities from a trained LSTM model."""
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = torch.FloatTensor(X[i:i+batch_size]).to(device)
            p = torch.sigmoid(model(batch)).cpu().numpy()
            probs.append(p)
    return np.concatenate(probs)

print(f"LSTM input features: {len(lstm_input_features)}")
print(f"Sequence length: {SEQ_LEN}, Epochs: {LSTM_EPOCHS}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [0]:
lstm_cv = load_checkpoint("cv3_lstm")
if lstm_cv is None:
    fold_results = []

    for i, fold in enumerate(cv_folds, 1):
        # Filter by year (same temporal splits as all other models)
        train_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        # Undersample train fold (same seed pattern as other models)
        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)
        train_size = train_balanced.count()

        # Convert Spark → pandas → sequences
        train_pdf = train_balanced.select(lstm_columns_to_collect).toPandas()
        test_pdf = test_fold.select(lstm_columns_to_collect).toPandas()
        train_pdf = train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
        test_pdf = test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

        print(f"\nFold {i}: {fold['label']}")
        print(f"  Building sequences (train={len(train_pdf)}, test={len(test_pdf)})...")
        X_tr, y_tr = build_sequences(train_pdf, SEQ_LEN, lstm_input_features, "DEP_DEL15")
        X_te, y_te = build_sequences(test_pdf, SEQ_LEN, lstm_input_features, "DEP_DEL15")

        # Train LSTM
        lstm_model, device, train_time = train_lstm(
            X_tr, y_tr, input_dim=len(lstm_input_features), epochs=LSTM_EPOCHS
        )

        # Predict on both train and test
        train_probs = predict_lstm(lstm_model, X_tr, device)
        test_probs = predict_lstm(lstm_model, X_te, device)
        train_preds = (train_probs >= 0.50).astype(float)
        test_preds = (test_probs >= 0.50).astype(float)

        # Evaluate (same metrics as all other models)
        train_metrics = evaluate_numpy(y_tr, train_preds, train_probs)
        test_metrics = evaluate_numpy(y_te, test_preds, test_probs)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_size, "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"  Training time: {train_time}s")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    lstm_cv = pd.DataFrame(fold_results)
    save_checkpoint(lstm_cv, "cv3_lstm")
else:
    pass

### Cross-Validation Results: Train vs. Validation Comparison

The chart below shows train and validation metrics side-by-side for each model across all three CV folds. The gap values above each bar pair indicate the degree of overfitting: larger positive gaps mean the model is memorizing training data rather than generalizing.

In [0]:
models = ["Logistic Regression", "Random Forest", "Gradient Boosted Trees", "MLP Shallow", "MLP Deep", "LSTM"]
model_data = {"Logistic Regression": lr_cv, "Random Forest": rf_cv, "Gradient Boosted Trees": gbt_cv, "MLP Shallow": mlp_cv, "MLP Deep": mlp_deep_cv, "LSTM": lstm_cv}
metrics = ["F2", "Recall", "Precision", "AUC-PR"]

fig, axes = plt.subplots(len(metrics), len(models), figsize=(5 * len(models), 4 * len(metrics)), sharey="row")

fold_labels = ["Fold 1", "Fold 2", "Fold 3"]
x = np.arange(len(fold_labels))
width = 0.3

for row, metric in enumerate(metrics):
    for col_idx, model_name in enumerate(models):
        ax = axes[row, col_idx]
        df = model_data[model_name]
        train_col = f"Train_{metric}"
        test_col = f"Test_{metric}"

        ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
        ax.bar(x + width/2, df[test_col], width, label="Validation", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(fold_labels, fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

        for j in range(len(df)):
            gap = df[train_col].iloc[j] - df[test_col].iloc[j]
            y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
            ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

        if col_idx == 0:
            ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(model_name)
        if row == 0 and col_idx == len(models) - 1:
            ax.legend(fontsize=8)

fig.suptitle("Phase III Rolling CV: Train vs Validation (Yearly Folds)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Final Evaluation: Train 2015 - 2018, Test 2019

After cross-validation, we train each model on the **full 2015 - 2018 training set** (with undersampling) and evaluate on the **2019 blind test set**. This provides the performance for each model on unseen future data. The best-performing model will be selected for hyperparameter tuning.

In [0]:
final_results_df = load_checkpoint("cv3_final_results_2019")
final_models = {}

# If checkpoint exists but is missing MLP Deep, run only MLP Deep and append
if final_results_df is not None and "MLP Deep" not in final_results_df["Model"].values:
    print("Checkpoint loaded but missing MLP Deep: running MLP Deep only...")
    train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)

    mlp_deep_result, mlp_deep_final = run_experiment(
        MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[mlp_input_dim, 128, 64, 2], maxIter=100, blockSize=128, seed=42
        ),
        mlp_pipeline_stages, train_balanced_full, otpw_test_df,
        "MLP Deep", "Train 2015-2018, Test 2019"
    )
    final_models["mlp_deep"] = mlp_deep_final

    # Append MLP Deep to existing results and re-save checkpoint
    final_results_df = pd.concat([final_results_df, pd.DataFrame([mlp_deep_result])], ignore_index=True)
    save_checkpoint(final_results_df, "cv3_final_results_2019")

elif final_results_df is None:
    train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)
    final_results = []
    final_models = {}

    # Logistic Regression
    lr_result, lr_final = run_experiment(
        LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
        lr_pipeline_stages, train_balanced_full, otpw_test_df,
        "Logistic Regression", "Train 2015-2018, Test 2019"
    )
    final_results.append(lr_result)
    final_models["lr"] = lr_final

    # Random Forest
    rf_result, rf_final = run_experiment(
        RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
        tree_pipeline_stages, train_balanced_full, otpw_test_df,
        "Random Forest", "Train 2015-2018, Test 2019"
    )
    final_results.append(rf_result)
    final_models["rf"] = rf_final

    # Gradient Boosted Trees
    gbt_result, gbt_final = run_experiment(
        GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
        tree_pipeline_stages, train_balanced_full, otpw_test_df,
        "Gradient Boosted Trees", "Train 2015-2018, Test 2019"
    )
    final_results.append(gbt_result)
    final_models["gbt"] = gbt_final

    # MLP
    mlp_result, mlp_final = run_experiment(
        MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[mlp_input_dim, 64, 2], maxIter=100, blockSize=128, seed=42
        ),
        mlp_pipeline_stages, train_balanced_full, otpw_test_df,
        "MLP Shallow", "Train 2015-2018, Test 2019"
    )
    final_results.append(mlp_result)
    final_models["mlp"] = mlp_final

    # MLP Deep (2 hidden layers)
    mlp_deep_result, mlp_deep_final = run_experiment(
        MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[mlp_input_dim, 128, 64, 2], maxIter=100, blockSize=128, seed=42
        ),
        mlp_pipeline_stages, train_balanced_full, otpw_test_df,
        "MLP Deep", "Train 2015-2018, Test 2019"
    )
    final_results.append(mlp_deep_result)
    final_models["mlp_deep"] = mlp_deep_final

    final_results_df = pd.DataFrame(final_results)
    save_checkpoint(final_results_df, "cv3_final_results_2019")
else:
    final_models = {}

In [0]:
# LSTM blind test: Train on full balanced 2015-2018, test on 2019
lstm_final_df = load_checkpoint("cv3_lstm_final_2019")
if lstm_final_df is None:
    train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)

    # Convert to pandas and build sequences
    train_pdf = train_balanced_full.select(lstm_columns_to_collect).toPandas()
    test_pdf = otpw_test_df.select(lstm_columns_to_collect).toPandas()
    train_pdf = train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
    test_pdf = test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

    print("Building train sequences...")
    X_train_lstm, y_train_lstm = build_sequences(train_pdf, SEQ_LEN, lstm_input_features, "DEP_DEL15")
    print(f"Train sequences: {X_train_lstm.shape}")
    print("Building test sequences...")
    X_test_lstm, y_test_lstm = build_sequences(test_pdf, SEQ_LEN, lstm_input_features, "DEP_DEL15")
    print(f"Test sequences: {X_test_lstm.shape}")

    # Train
    lstm_final_model, device, lstm_train_time = train_lstm(
        X_train_lstm, y_train_lstm, input_dim=len(lstm_input_features), epochs=LSTM_EPOCHS
    )

    # Predict
    train_probs = predict_lstm(lstm_final_model, X_train_lstm, device)
    test_probs = predict_lstm(lstm_final_model, X_test_lstm, device)
    train_preds = (train_probs >= 0.50).astype(float)
    test_preds = (test_probs >= 0.50).astype(float)

    # Evaluate
    train_metrics = evaluate_numpy(y_train_lstm, train_preds, train_probs)
    test_metrics = evaluate_numpy(y_test_lstm, test_preds, test_probs)

    print(f"\n============================================================")
    print(f"LSTM on Train 2015-2018, Test 2019")
    print(f"Training time: {lstm_train_time}s")
    print(f"\n============================================================")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"------------------------------------------")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    lstm_result = {
        "Model": "LSTM",
        "Dataset": "Train 2015-2018, Test 2019",
        "Train Time (s)": lstm_train_time,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }
    lstm_final_df = pd.DataFrame([lstm_result])
    save_checkpoint(lstm_final_df, "cv3_lstm_final_2019")
else:
    lstm_result = lstm_final_df.iloc[0].to_dict()

# Append LSTM row to the final results table
final_results_df = pd.concat([final_results_df, pd.DataFrame([lstm_result])], ignore_index=True)

In [0]:
# Display blind test results and identify best model for tuning
final_results_df["Overfit_Gap"] = (final_results_df["Train_F2"] - final_results_df["Test_F2"]).round(4)
print("2019 Blind Test Results (all models):")
display(final_results_df[["Model", "Test_F2", "Test_Recall", "Test_Precision", "Test_AUC-PR", "Train_F2", "Overfit_Gap", "Train Time (s)"]].round(4))

best_model = final_results_df.sort_values("Test_F2", ascending=False).iloc[0]
print(f"\nBest model by Test F2: {best_model['Model']} with F2 = {best_model['Test_F2']:.4f}")

### Threshold Tunning

#### Threshold Tuning (Logistic Regression)

By default, PySpark use a 0.5 probability threshold: if the model assigns > 50% chance of delay, it predicts "delayed." However, since F2 weights recall twice as much as precision, a lower threshold can improve F2 by catching more delays at the cost of more false alarms.

We sweep thresholds from 0.20 to 0.55 on Logistic Regression and compute F2, Recall, and Precision at each threshold. The model itself is not retrained but only the decision boundary changes. This is to answer a more operational decision rather than model performance itself: "how confident should the model be before we flag a flight?"

In [0]:
from pyspark.sql.functions import when, udf
from pyspark.sql.types import DoubleType

threshold_df = load_checkpoint("threshold_tuning_lr")

if threshold_df is None:
    if "lr" not in final_models or final_models.get("lr") is None:
        train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)
        lr_model = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20)
        lr_full_pipeline = Pipeline(stages=lr_pipeline_stages + [lr_model])
        lr_fitted = lr_full_pipeline.fit(train_balanced_full)
    else:
        lr_fitted = final_models["lr"]

    lr_test_preds = lr_fitted.transform(otpw_test_df)

    get_prob_delayed = udf(lambda v: float(v[1]), DoubleType())
    lr_test_preds = lr_test_preds.withColumn("prob_delayed", get_prob_delayed(col("probability")))

    thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]
    threshold_results = []

    for t in thresholds:
        reclassified = lr_test_preds.withColumn(
            "prediction", when(col("prob_delayed") >= t, 1.0).otherwise(0.0)
        )
        metrics = evaluate_model(reclassified)
        threshold_results.append({
            "Threshold": t,
            "F2": metrics["F2"],
            "Recall": metrics["Recall"],
            "Precision": metrics["Precision"],
            "AUC-PR": metrics["AUC-PR"],
        })
        print(f"threshold={t:.2f}: F2={metrics['F2']:.4f}, Recall={metrics['Recall']:.4f}, Precision={metrics['Precision']:.4f}")

    threshold_df = pd.DataFrame(threshold_results)
    save_checkpoint(threshold_df, "threshold_tuning_lr")

display(threshold_df.round(4))


#### Threshold Tuning (Gradient Boosted Trees)

We will apply the similar logic to GBT

In [0]:
gbt_threshold_df = load_checkpoint("threshold_tuning_gbt")

if gbt_threshold_df is None:
    if "gbt" not in final_models or final_models.get("gbt") is None:
        train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)
        gbt_model = GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42)
        gbt_full_pipeline = Pipeline(stages=tree_pipeline_stages + [gbt_model])
        gbt_fitted = gbt_full_pipeline.fit(train_balanced_full)
    else:
        gbt_fitted = final_models["gbt"]

    gbt_test_preds = gbt_fitted.transform(otpw_test_df)
    gbt_test_preds = gbt_test_preds.withColumn("prob_delayed", get_prob_delayed(col("probability")))

    thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]
    gbt_threshold_results = []

    for t in thresholds:
        reclassified = gbt_test_preds.withColumn(
            "prediction", when(col("prob_delayed") >= t, 1.0).otherwise(0.0)
        )
        metrics = evaluate_model(reclassified)
        gbt_threshold_results.append({
            "Threshold": t,
            "F2": metrics["F2"],
            "Recall": metrics["Recall"],
            "Precision": metrics["Precision"],
            "AUC-PR": metrics["AUC-PR"],
        })
        print(f"threshold={t:.2f}: F2={metrics['F2']:.4f}, Recall={metrics['Recall']:.4f}, Precision={metrics['Precision']:.4f}")

    gbt_threshold_df = pd.DataFrame(gbt_threshold_results)
    save_checkpoint(gbt_threshold_df, "threshold_tuning_gbt")

display(gbt_threshold_df.round(4))


#### Threshold Tunning Result

In [0]:
# Plot threshold tuning results for both LR and GBT
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model_name, tdf in zip(axes, ["Logistic Regression", "Gradient Boosted Trees"], [threshold_df, gbt_threshold_df]):
    ax.plot(tdf["Threshold"], tdf["F2"], "o-", color="tab:blue", linewidth=2, label="F2 Score")
    ax.plot(tdf["Threshold"], tdf["Recall"], "s--", color="tab:green", linewidth=1.5, label="Recall")
    ax.plot(tdf["Threshold"], tdf["Precision"], "^--", color="tab:red", linewidth=1.5, label="Precision")

    best_idx = tdf["F2"].idxmax()
    best_t = tdf.loc[best_idx, "Threshold"]
    best_f2 = tdf.loc[best_idx, "F2"]
    ax.axvline(x=best_t, color="gray", linestyle=":", alpha=0.7)
    ax.annotate(f"Best: t={best_t:.2f}\nF2={best_f2:.4f}",
                xy=(best_t, best_f2), xytext=(best_t + 0.05, best_f2 - 0.05),
                arrowprops=dict(arrowstyle="->", color="gray"), fontsize=10, color="tab:blue")

    ax.set_xlabel("Classification Threshold", fontsize=12)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"{model_name}", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.15, 0.60)

fig.suptitle("Threshold Tuning: LR vs GBT", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print summary for both models
lr_best = threshold_df.loc[threshold_df["F2"].idxmax()]
gbt_best = gbt_threshold_df.loc[gbt_threshold_df["F2"].idxmax()]
lr_default = threshold_df[threshold_df["Threshold"] == 0.50].iloc[0]
gbt_default = gbt_threshold_df[gbt_threshold_df["Threshold"] == 0.50].iloc[0]

print(f"Logistic Regression:")
print(f"  Best threshold: {lr_best['Threshold']:.2f} - F2={lr_best['F2']:.4f}, Recall={lr_best['Recall']:.4f}, Precision={lr_best['Precision']:.4f}")
print(f"  vs default 0.50:  F2={lr_default['F2']:.4f} (improvement: {lr_best['F2'] - lr_default['F2']:+.4f})")
print(f"\nGradient Boosted Trees:")
print(f"  Best threshold: {gbt_best['Threshold']:.2f} - F2={gbt_best['F2']:.4f}, Recall={gbt_best['Recall']:.4f}, Precision={gbt_best['Precision']:.4f}")
print(f"  vs default 0.50:  F2={gbt_default['F2']:.4f} (improvement: {gbt_best['F2'] - gbt_default['F2']:+.4f})")

winner = "Logistic Regression" if lr_best["F2"] >= gbt_best["F2"] else "Gradient Boosted Trees"
print(f"\nBest model at optimal threshold: {winner} (F2={max(lr_best['F2'], gbt_best['F2']):.4f})")

## Model Selection Summary

We evaluated four models through cross-validation, blind test on 2019, and threshold tuning. Below are results:

### Blind Test Results (2019, default threshold = 0.50)

| Model | Train F2 | Test F2 | Overfit Gap | Test Recall | Test Precision | AUC-PR | Train Time |
|-------|----------|---------|-------------|-------------|----------------|--------|------------|
| **LR (Baseline)** | 0.6326 | **0.5047** | 0.1279 | 0.6168 | 0.2923 | 0.3551 | 30s |
| **GBT** | 0.6196 | 0.4974 | 0.1222 | 0.5910 | 0.3045 | **0.3799** | 400s |
| RF | 0.6032 | 0.4903 | 0.1129 | 0.5893 | 0.2932 | 0.3528 | 115s |
| MLP | 0.5436 | 0.4698 | 0.0738 | 0.5659 | 0.2798 | 0.3384 | 278s |

### With Threshold Tuning

| Model | Best Threshold | Test F2 | Test Recall | Test Precision | F2 Improvement |
|-------|---------------|---------|-------------|----------------|----------------|
| **GBT** | 0.35 | **0.5598** | 0.8787 | 0.2283 | +0.0624 |
| LR | 0.35 | 0.5562 | 0.8777 | 0.2256 | +0.0515 |

### Key findings

1. **Overfitting is moderate across all models** (train F2 ~0.12 higher than test F2). MLP shows the smallest gap (0.074) but also the worst absolute performancd.
2. **At default threshold (0.50):** LR leads with F2=0.5047. All four models are within 0.035 of each other.
3. **With threshold tuning:** GBT overtakes LR (0.5598 vs 0.5562). GBT's higher AUC-PR (0.3799 vs 0.3551) makes it a slight better when threshold is optimized for F2. Both models peak at threshold 0.35 with ~88% recall and ~23% precision.

**Selected for hyperparameter tuning:** We proceed with **Gradient Boosted Trees** as the primary model for grid search, given its best threshold-tuned F2 (0.5598) and highest AUC-PR. LR remains the baseline reference. RF and MLP are excluded.

## Hyperparameter Tuning

With GBT and LR identified as the two best models, we now tune their hyperparameters to maximize F2 on the 2019 blind test. For each configuration, we train on the full balanced 2015-2018 set and evaluate at both the default threshold (0.50) and the optimal threshold (0.35). Train and test F2 are reported for overfitting diagnosis.

### GBT Grid Search

GBT has three key hyperparameters that control model complexity and learning speed:

| Parameter | Values | What it controls |
|-----------|--------|------------------|
| `maxIter` | [20, 50, 100] | Number of boosting rounds, more rounds allow the model to correct more errors but risk overfitting |
| `maxDepth` | [3, 5, 8] | Tree depth, deeper trees capture more complex feature interactions |
| `stepSize` | [0.05, 0.1] | Learning rate, smaller values make each tree contribute less, requiring more iterations but often improving generalization |

Total: 3 x 3 x 2 = 18 configurations.

In [0]:
gs_df = load_checkpoint("grid_search_gbt")
if gs_df is None:
    # Prepare balanced training data
    train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)

    param_grid = [
        {"maxIter": mi, "maxDepth": md, "stepSize": ss}
        for mi in [20, 50, 100]
        for md in [3, 5, 8]
        for ss in [0.05, 0.1]
    ]

    gs_results = []
    best_threshold = 0.35

    for idx, params in enumerate(param_grid, 1):
        gbt_gs = GBTClassifier(
            featuresCol="features", labelCol=label_col,
            maxIter=params["maxIter"], maxDepth=params["maxDepth"],
            stepSize=params["stepSize"], seed=42
        )
        full_pipeline = Pipeline(stages=tree_pipeline_stages + [gbt_gs])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced_full)
        train_time = round(time.time() - start, 1)

        # Evaluate at default threshold (0.50)
        train_preds = fitted.transform(train_balanced_full)
        test_preds = fitted.transform(otpw_test_df)
        train_metrics = evaluate_model(train_preds)
        test_metrics_default = evaluate_model(test_preds)

        # Evaluate at optimal threshold (0.35)
        test_preds = test_preds.withColumn("prob_delayed", get_prob_delayed(col("probability")))
        test_preds_tuned = test_preds.withColumn(
            "prediction", when(col("prob_delayed") >= best_threshold, 1.0).otherwise(0.0)
        )
        test_metrics_tuned = evaluate_model(test_preds_tuned)

        gs_results.append({
            "maxIter": params["maxIter"],
            "maxDepth": params["maxDepth"],
            "stepSize": params["stepSize"],
            "Train_F2": train_metrics["F2"],
            "Test_F2_t50": test_metrics_default["F2"],
            "Test_F2_t35": test_metrics_tuned["F2"],
            "Test_Recall_t35": test_metrics_tuned["Recall"],
            "Test_Precision_t35": test_metrics_tuned["Precision"],
            "Test_AUC-PR": test_metrics_default["AUC-PR"],
            "Overfit_Gap": round(train_metrics["F2"] - test_metrics_default["F2"], 4),
            "Train_Time_s": train_time,
        })

        print(
            f"[{idx}/{len(param_grid)}] maxIter={params['maxIter']:>3}, maxDepth={params['maxDepth']}, stepSize={params['stepSize']} | "
            f"Train F2={train_metrics['F2']:.4f}, Test F2(t=0.50)={test_metrics_default['F2']:.4f}, "
            f"Test F2(t=0.35)={test_metrics_tuned['F2']:.4f} | {train_time}s"
        )

    gs_df = pd.DataFrame(gs_results).sort_values("Test_F2_t35", ascending=False).reset_index(drop=True)
    save_checkpoint(gs_df, "grid_search_gbt")

print("\nGrid Search Results (sorted by Test F2 at t=0.35):")
display(gs_df.round(4))

In [0]:
# GBT grid search summary
best_gbt = gs_df.iloc[0]
gbt_baseline_f2 = 0.5598

print("GBT Grid Search - Best configuration:")
print(f"  maxIter={int(best_gbt['maxIter'])}, maxDepth={int(best_gbt['maxDepth'])}, stepSize={best_gbt['stepSize']}")
print(f"  Train F2:        {best_gbt['Train_F2']:.4f}")
print(f"  Test F2 (t=0.50): {best_gbt['Test_F2_t50']:.4f}")
print(f"  Test F2 (t=0.35): {best_gbt['Test_F2_t35']:.4f}")
print(f"  Overfit Gap:     {best_gbt['Overfit_Gap']:+.4f}")
print(f"  AUC-PR:          {best_gbt['Test_AUC-PR']:.4f}")
print(f"  vs baseline GBT: F2 change at t=0.35: {best_gbt['Test_F2_t35'] - gbt_baseline_f2:+.4f}")

### LR Grid Search

Logistic Regression has fewer tunable parameters but regularization can impact performance, especially with high-dimensional OHE features:

| Parameter | Values | What it controls |
|-----------|--------|------------------|
| `regParam` | [0.001, 0.01, 0.1] | Regularization strength: higher values penalize large coefficients more, reducing overfitting |
| `elasticNetParam` | [0.0, 0.5] | Mixing parameter: 0.0 = pure Ridge (L2), 0.5 = equal mix of L1 and L2 (encourages sparsity) |

Total: 3 x 2 = 6 configurations.

In [0]:
lr_gs_df = load_checkpoint("grid_search_lr")
if lr_gs_df is None:
    train_balanced_full = undersample_majority(otpw_train_df, label_col=label_col, seed=42)

    lr_param_grid = [
        {"regParam": rp, "elasticNetParam": en}
        for rp in [0.001, 0.01, 0.1]
        for en in [0.0, 0.5]
    ]

    lr_gs_results = []
    best_threshold = 0.35

    for idx, params in enumerate(lr_param_grid, 1):
        lr_gs_model = LogisticRegression(
            featuresCol="features", labelCol=label_col, maxIter=20,
            regParam=params["regParam"], elasticNetParam=params["elasticNetParam"]
        )
        full_pipeline = Pipeline(stages=lr_pipeline_stages + [lr_gs_model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced_full)
        train_time = round(time.time() - start, 1)

        train_preds = fitted.transform(train_balanced_full)
        test_preds = fitted.transform(otpw_test_df)
        train_metrics = evaluate_model(train_preds)
        test_metrics_default = evaluate_model(test_preds)

        test_preds = test_preds.withColumn("prob_delayed", get_prob_delayed(col("probability")))
        test_preds_tuned = test_preds.withColumn(
            "prediction", when(col("prob_delayed") >= best_threshold, 1.0).otherwise(0.0)
        )
        test_metrics_tuned = evaluate_model(test_preds_tuned)

        lr_gs_results.append({
            "regParam": params["regParam"],
            "elasticNetParam": params["elasticNetParam"],
            "Train_F2": train_metrics["F2"],
            "Test_F2_t50": test_metrics_default["F2"],
            "Test_F2_t35": test_metrics_tuned["F2"],
            "Test_Recall_t35": test_metrics_tuned["Recall"],
            "Test_Precision_t35": test_metrics_tuned["Precision"],
            "Test_AUC-PR": test_metrics_default["AUC-PR"],
            "Overfit_Gap": round(train_metrics["F2"] - test_metrics_default["F2"], 4),
            "Train_Time_s": train_time,
        })

        print(
            f"[{idx}/{len(lr_param_grid)}] regParam={params['regParam']}, elasticNetParam={params['elasticNetParam']} | "
            f"Train F2={train_metrics['F2']:.4f}, Test F2(t=0.50)={test_metrics_default['F2']:.4f}, "
            f"Test F2(t=0.35)={test_metrics_tuned['F2']:.4f} | {train_time}s"
        )

    lr_gs_df = pd.DataFrame(lr_gs_results).sort_values("Test_F2_t35", ascending=False).reset_index(drop=True)
    save_checkpoint(lr_gs_df, "grid_search_lr")

print("\nLR Grid Search Results (sorted by Test F2 at t=0.35):")
display(lr_gs_df.round(4))

In [0]:
# Combined hyperparameter tuning summary
best_lr = lr_gs_df.iloc[0]
lr_baseline_f2 = 0.5562

print("LR Grid Search - Best configuration:")
print(f"  regParam={best_lr['regParam']}, elasticNetParam={best_lr['elasticNetParam']}")
print(f"  Train F2:        {best_lr['Train_F2']:.4f}")
print(f"  Test F2 (t=0.50): {best_lr['Test_F2_t50']:.4f}")
print(f"  Test F2 (t=0.35): {best_lr['Test_F2_t35']:.4f}")
print(f"  Overfit Gap:     {best_lr['Overfit_Gap']:+.4f}")
print(f"  vs baseline LR:  F2 change at t=0.35: {best_lr['Test_F2_t35'] - lr_baseline_f2:+.4f}")

print(f"\n============================================================")
print(f"Hyperparameter Tuning Summary:")
print(f"  Best GBT: F2={best_gbt['Test_F2_t35']:.4f} (maxIter={int(best_gbt['maxIter'])}, maxDepth={int(best_gbt['maxDepth'])}, stepSize={best_gbt['stepSize']})")
print(f"  Best LR:  F2={best_lr['Test_F2_t35']:.4f} (regParam={best_lr['regParam']}, elasticNetParam={best_lr['elasticNetParam']})")
winner = "GBT" if best_gbt['Test_F2_t35'] >= best_lr['Test_F2_t35'] else "LR"
print(f"  Winner:   {winner} (F2={max(best_gbt['Test_F2_t35'], best_lr['Test_F2_t35']):.4f})")

In [0]:
best_lr = lr_gs_df.iloc[0]
lr_baseline_f2_t50 = 0.5047
lr_baseline_f2_t35 = 0.5562
gbt_baseline_f2_t50 = 0.4974
gbt_baseline_f2_t35 = 0.5598

print(f"============================================================")
print("Hyperparameter Tuning Results vs Baselines")
print(f"============================================================")

print(f"\n{'Experiment':<40} {'Train F2':>10} {'Test F2':>12} {'Test F2':>12} {'Overfit':>10}")
print(f"{'':40} {'':>10} {'(t=0.50)':>12} {'(t=0.35)':>12} {'Gap':>10}")
print("------------------------------------------------------------")
print(f"{'LR baseline (default params)':<40} {'0.6326':>10} {lr_baseline_f2_t50:>12.4f} {lr_baseline_f2_t35:>12.4f} {'0.1279':>10}")
lr_label = f"LR tuned (reg={best_lr['regParam']}, en={best_lr['elasticNetParam']})"
print(f"{lr_label:<40} {best_lr['Train_F2']:>10.4f} {best_lr['Test_F2_t50']:>12.4f} {best_lr['Test_F2_t35']:>12.4f} {best_lr['Overfit_Gap']:>+10.4f}")
print(f"{'  Improvement':<40} {'':>10} {best_lr['Test_F2_t50'] - lr_baseline_f2_t50:>+12.4f} {best_lr['Test_F2_t35'] - lr_baseline_f2_t35:>+12.4f}")
print()
print(f"{'GBT baseline (default params)':<40} {'0.6196':>10} {gbt_baseline_f2_t50:>12.4f} {gbt_baseline_f2_t35:>12.4f} {'0.1222':>10}")
gbt_label = f"GBT tuned (iter={int(best_gbt['maxIter'])},d={int(best_gbt['maxDepth'])},lr={best_gbt['stepSize']})"
print(f"{gbt_label:<40} {best_gbt['Train_F2']:>10.4f} {best_gbt['Test_F2_t50']:>12.4f} {best_gbt['Test_F2_t35']:>12.4f} {best_gbt['Overfit_Gap']:>+10.4f}")
print(f"{'  Improvement':<40} {'':>10} {best_gbt['Test_F2_t50'] - gbt_baseline_f2_t50:>+12.4f} {best_gbt['Test_F2_t35'] - gbt_baseline_f2_t35:>+12.4f}")
print()
print(f"\n============================================================")
winner = "GBT" if best_gbt['Test_F2_t35'] >= best_lr['Test_F2_t35'] else "LR"
print(f"Best overall: {winner} with F2={max(best_gbt['Test_F2_t35'], best_lr['Test_F2_t35']):.4f} at t=0.35")
print(f"\nKey insight: Hyperparameter tuning improved GBT by {best_gbt['Test_F2_t35'] - gbt_baseline_f2_t35:+.4f} and LR by {best_lr['Test_F2_t35'] - lr_baseline_f2_t35:+.4f}.")

## Results & Discussion

### Experimentation Progress: Phase II to Phase III

The table below traces our F2-score improvement across the entire project. Phase II experiments isolate the impact of feature engineering on a 1-year dataset. Phase III scales to 4 years with new features, then independently tunes both LR and GBT through threshold and hyperparameter optimization. The V2 extended feature set (weather trends + tail-number delay propagation) produces the largest single improvement, pushing GBT from F2 = 0.56 to F2 = 0.62 at the default threshold and F2 = 0.63 at t=0.35.

### Phase II: Baseline + Feature Engineering (1-year OTPW, 2015, Train Q1-Q4, Test = Q4)

| Step | Features | LR F2 | LR Delta | RF F2 | RF Delta |
|------|----------|-------|------|-------|------|
| 1. Raw Baseline | Raw weather, distance, carrier, airport, time | 0.4868 | — | 0.4594 | — |
| 2. + Basic FE | Standardized weather, PrecipBinary, VisibilityCat, weekend, route | 0.4875 | +0.0007 | 0.4504 | -0.0090 |
| 3. + Time-Series FE (RFM) | Origin delay freq, carrier delay freq, days since delay | 0.4674 | -0.0201 | 0.4349 | -0.0155 |

**Phase II finding:** Basic feature engineering had negligible impact (<1%). RFM time-series features improved precision but hurt recall, reducing overall F2. The bottleneck was not transformations — it was data scale and feature signal.

### Phase III: Model Comparison, Tuning, and Extended Features (4-year OTPW, Train 2015-2018, Test = 2019)

| Step | What Changed | Best Model | F2 (Blind Test) | Delta |
|------|-------------|-----------|-----------------|-------|
| 4. Scale to 4yr + New FE | PageRank, congestion (2-4hr), 2hr weather lag | GBT | 0.4974 | — |
| 5. + Threshold Tuning (t=0.35) | Lower decision boundary from 0.50 to 0.35 | GBT | 0.5598 | +0.0624 |
| 6. + Hyperparameter Tuning | GBT: iter=100, depth=8, lr=0.1 | GBT | 0.5629 | +0.0031 |
| 7. + V2 Features (Trend + Tail #) | Weather trends, tail-number delay propagation | GBT (t=0.50) | 0.6198 | +0.0569 |
| 7b. V2 GBT at t=0.35 | Same V2 features, operational threshold | GBT (t=0.35) | **0.6336** | +0.0707 |
| 8. + LSTM HP Tuning | Grid search on hidden_dim, layers, dropout | LSTM (t=0.35) | TBD | — |
| 9. + V3 Features (CV only) | Add absolute 2h weather + VisibilityCat | GBT/LSTM | CV validated | — |

### Summary

| Improvement Source | Impact on F2 |
|-------------------|-------------|
| Phase II FE (steps 2-3) | Negligible |
| Phase III scale + new FE (step 4) | Baseline established |
| Threshold tuning (step 5) | **+0.0624** |
| Hyperparameter tuning (step 6) | +0.0031 |
| V2 feature engineering (step 7) | **+0.0569 (t=0.50), +0.0707 (t=0.35)** |
| V3 features (step 9) | Cross-validated; blind test pending |

Feature engineering (V2 extended features) and operational threshold design together account for the vast majority of all improvement. Hyperparameter tuning was near-irrelevant (+0.003). This confirms that in applied ML, the data you build matters more than the model you choose.


In [0]:
# Experimentation Progression Waterfall
fig, ax2 = plt.subplots(1, 1, figsize=(10, 6))

steps = [
    "Phase II\nRaw Baseline",
    "+ Basic FE",
    "+ Time-Series\n(RFM)",
    "Phase III\n4yr + New FE",
    "+ Threshold\n(t=0.35)",
    "+ HP Tuning\n(GBT)"
]
f2_values = [0.4868, 0.4875, 0.4674, 0.5047, 0.5598, 0.5629]
deltas = [0] + [f2_values[i] - f2_values[i-1] for i in range(1, len(f2_values))]
colors = ["#4472C4"] + ["#2E7D32" if d > 0.001 else "#C62828" if d < -0.001 else "#9E9E9E" for d in deltas[1:]]

bars = ax2.bar(range(len(steps)), f2_values, color=colors, edgecolor="white", linewidth=1.5)

for i, (bar, val, delta) in enumerate(zip(bars, f2_values, deltas)):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
    if i > 0:
        delta_color = "#2E7D32" if delta > 0.001 else "#C62828" if delta < -0.001 else "#9E9E9E"
        ax2.text(bar.get_x() + bar.get_width()/2, val - 0.025, f"{delta:+.4f}", ha="center", fontsize=8, color=delta_color)

ax2.axvline(x=2.5, color="gray", linestyle="--", alpha=0.5)
ax2.text(1.0, 0.58, "Phase II (2015)", ha="center", fontsize=9, color="gray", style="italic")
ax2.text(4.0, 0.58, "Phase III (2015 - 2019)", ha="center", fontsize=9, color="gray", style="italic")

ax2.set_xticks(range(len(steps)))
ax2.set_xticklabels(steps, fontsize=9)
ax2.set_ylabel("F2 Score", fontsize=11)
ax2.set_title("Experimentation Progression: Where the Gains Came From", fontsize=13)
ax2.set_ylim(0.40, 0.60)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

### Model Comparison on 2019 Blind Test

The progression above shows that scaling data and tuning thresholds drove the largest gains. But how do all six models compare head-to-head on the 2019 blind test? The chart below shows train and test F2 side-by-side for each model at default threshold (t=0.50), with the overfitting gap labeled above each pair.

In [0]:
_main_df = load_checkpoint("cv3_final_results_2019")
_lstm_df = load_checkpoint("cv3_lstm_final_2019")

if _main_df is not None and _lstm_df is not None:
    _all_results = pd.concat([_main_df, _lstm_df], ignore_index=True)
elif _main_df is not None:
    _all_results = _main_df
else:
    _all_results = final_results_df

model_names = ["LR", "RF", "GBT", "MLP\n(1 layer)", "MLP\n(2 layers)", "LSTM"]
_model_keys = ["Logistic Regression", "Random Forest", "Gradient Boosted Trees", "MLP Shallow", "MLP Deep", "LSTM"]
train_f2 = []
test_f2 = []

for name in _model_keys:
    row = _all_results[_all_results["Model"] == name]
    if len(row) > 0:
        train_f2.append(float(row["Train_F2"].values[0]))
        test_f2.append(float(row["Test_F2"].values[0]))
    else:
        train_f2.append(0.0)
        test_f2.append(0.0)
        print(f"WARNING: No results found for {name}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x = np.arange(len(model_names))
width = 0.3
ax.bar(x - width/2, train_f2, width, label="Train F2", color="steelblue")
ax.bar(x + width/2, test_f2, width, label="Test F2", color="darkorange")

for i in range(len(model_names)):
    gap = train_f2[i] - test_f2[i]
    y_top = max(train_f2[i], test_f2[i])
    if y_top > 0:
        ax.text(x[i], y_top + 0.01, f"gap:{gap:+.3f}", ha="center", fontsize=8, color="gray")

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylabel("F2 Score", fontsize=11)
ax.set_title("Train vs Test F2 - All Models on 2019 Blind Test (t=0.50)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 0.75)
plt.tight_layout()
plt.show()

### Key Findings

1. **Threshold tuning delivered the largest single improvement.** Lowering the decision boundary from 0.50 to 0.35 increased F2 by +0.06 for GBT and +0.05 for LR. This is not a model improvement, it is an operational decision to catch more delays at the cost of more false alarms. For an airline OCC, this trade-off is justified: missed delays are far more expensive than pre-allocated standby resources.

2. **New feature families drove the real improvement** In Phase II, standard transformations (standardization, binary tranformation, visibility bucketing) on 1-year data produced near-zero improvement (+0.0007). In Phase III, scaling to 4 years and introducing new feature families: graph-based PageRank scores, short-term airport congestion (2-4hr delay counts), and 2-hour weather lag, improved F2 by +0.017 over the raw baseline. The improvement came from adding new predictive signal (airport importance, real-time congestion), not from transforming existing columns.

3. **Hyperparameter tuning was near-irrelevant.** GBT grid search (18 configurations) improved F2 by only +0.003. LR grid search produced zero improvement. The default parameters were already near-optimal, confirming that the performance bottleneck is feature signal, not model complexity.

4. **All models converged to similar F2 at default threshold.** LR (0.505), GBT (0.497), RF (0.490), and MLP (0.470) are all within a 0.035 range. This suggests the available features impose a ceiling that no single model architecture can break through at t=0.50.

### Neural Network Discussion (MLP & LSTM)

The client requested a neural network model. We implemented two MLP architectures using PySpark MLlib and an LSTM using PyTorch:

- **MLP (1 hidden layer, [input, 64, 2]):** Achieved the lowest F2 (0.470) among all models. The high-cardinality `ORIGIN` feature (300+ airports) was replaced with continuous PageRank scores to avoid dimensionality issues in dense layers, but this substitution lost route-specific information that tree models capture via OHE.
- **MLP (2 hidden layers, [input, 128, 64, 2]):** F2 (0.5029) Adding depth did not improve performance, suggesting the bottleneck is feature signal, not model capacity.
- **LSTM (extra credit):** Sequences of 5 flights per aircraft (`TAIL_NUM`) were used to model delay propagation. LSTM can theoretically capture cascading delays across an aircraft's daily legs, but the limited numeric feature set (same as MLP) constrains its advantage.

**Takeaway:** Neural networks did not outperform tree-based models.

### Gap Analysis: Why Is F2 Capped at ~0.56?

Our best V1 model achieves F2 = 0.5629 with 88% recall and 23% precision at t=0.35. The recall is good, we catch most delays. The bottleneck is **precision**: for every 4 flights we flag as delayed, only 1 actually is.

Two possible reasons:

1. **Undersampling inflates false positives at test time.** We train on a 50/50 balanced dataset, but the real test data is 80/20 (on-time/delayed). The model learns that delays are common (50%), but at test time only 20% of flights are delayed, so it over-flags on-time flights.

2. **Missing upstream propagation features.** Our V1 features capture airport-level congestion but not aircraft-specific propagation (is this specific plane running late from its previous leg?) or crew-level constraints (is the crew about to time out?).

These observations directly motivate the V2 feature engineering in the next section, which targets factor #3 by adding tail-number delay propagation and weather trend features.


# V2 Feature Set (Trend features) and LSTM Hyperparameter Tuning

The results above establish Gradient Boosted Trees with threshold tuning as the strongest pipeline among the six baseline models. To probe whether further gains are achievable, two additional experiments are conducted:

1. **Extended feature set.** All six models are retrained on an enriched dataset that adds two new feature families on top of the previous engineering work:
   - **Weather trend features (6h - 2h):** `DryBulbTemperature_trend_2h_6h`, `WindSpeed_trend_2h_6h`, `RelativeHumidity_trend_2h_6h`, and `Visibility_trend_2h_6h` (with standardized variants for the linear and neural models). A static weather snapshot misses the directionality of conditions; the trend features capture whether weather is worsen, which is a stronger predictor of delay than the current observation alone.
   - **Tail-number delay propagation:** `prev_DEP_DEL15` (whether the same aircraft's previous leg was delayed), `turnaround_minutes` (the gap between the previous arrival and the current scheduled departure). These capture the cascading effect.
2. **LSTM hyperparameter tuning.** The default LSTM configuration gives a competitive recall in the baseline comparison. We will do a grid search on LSTM to further exaimine the results.

To preserve the reproducibility of the baseline results, all experiments in this section read from a separate feature-engineering checkpoint (`041726_otpw_train_df`) and write to dedicated result files. The previous results remain unchanged from the earlier sections for later comparison between the experiments.


### Loading the Extended Feature-Engineering Checkpoint

The new train and test parquets contain the full set of original features plus the weather-trend and tail-number columns described above. They are loaded into separate DataFrames so that the original training and test sets remain available for downstream comparison.


In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
from pyspark.sql.functions import regexp_replace, mean, stddev, log1p, concat_ws
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

CHECKPOINT_DATE_V2 = "041726"

TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df_v2 = spark.read.parquet(f"{TEAM_DIR}/{CHECKPOINT_DATE_V2}_otpw_train_df.parquet")
otpw_test_df_v2 = spark.read.parquet(f"{TEAM_DIR}/{CHECKPOINT_DATE_V2}_otpw_test_df.parquet")

otpw_train_df_v2 = otpw_train_df_v2.withColumn("DEP_HOUR", hour("sched_depart_date_time")) \
                                   .withColumn("DISTANCE", col("DISTANCE").cast("double"))
otpw_test_df_v2 = otpw_test_df_v2.withColumn("DEP_HOUR", hour("sched_depart_date_time")) \
                                 .withColumn("DISTANCE", col("DISTANCE").cast("double"))

print(f"Train (2015 - 2018): {otpw_train_df_v2.count():,} rows x {len(otpw_train_df_v2.columns)} columns")
print(f"Test  (2019):       {otpw_test_df_v2.count():,} rows x {len(otpw_test_df_v2.columns)} columns")


### Pipeline for the Extended Feature Set

Each model receives a feature set tailored to its inductive bias, mirroring the design used earlier in the notebook:

- **Logistic Regression, MLP, and LSTM** consume standardized weather-trend features (`*_trend_2h_6h_stdized`) so that gradient updates are not dominated by features with larger natural scales.
- **Tree-based models (Random Forest and GBT)** consume the raw weather-trend features.
- **MLP and LSTM** continue to drop the high-cardinality `ORIGIN` one-hot encoding and rely on the continuous PageRank features instead.
- The categorical `VisibilityCat` feature is removed in this section because the continuous visibility-trend feature is the same.

Tail-number features (`prev_DEP_DEL15`, `turnaround_minutes`) are added to all models.


In [0]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Shared categorical features (used by LR and tree-based models)
# VisibilityCat is dropped here
categorical_features_v2 = [
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR",
]

indexers_v2 = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features_v2
]
encoders_v2 = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features_v2
]

# 1. Logistic Regression Pipeline (Extended)
#    Adds standardized weather-trend features (6h - 2h direction of change)
#    and tail-number delay-propagation features on top of the baseline LR set.
lr_numeric_features_v2 = [
    # Weather (standardized 6h - 2h trends)
    "DryBulbTemperature_trend_2h_6h_stdized",
    "WindSpeed_trend_2h_6h_stdized",
    "RelativeHumidity_trend_2h_6h_stdized",
    "Visibility_trend_2h_6h_stdized",
    "PrecipBinary",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (log of total flights replaces raw count)
    "delay_pct_flights_2_4hr_before", "log_total_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
    # Tail-number delay propagation
    "prev_DEP_DEL15", "turnaround_minutes",
]

lr_assembler_inputs_v2 = lr_numeric_features_v2 + [f"{c}_ohe" for c in categorical_features_v2]
lr_assembler_v2 = VectorAssembler(inputCols=lr_assembler_inputs_v2, outputCol="features", handleInvalid="skip")
lr_pipeline_stages_v2 = indexers_v2 + encoders_v2 + [lr_assembler_v2]

# 2. Tree-based Pipeline (RF, GBT)
#    Uses raw (un-standardized) trends
tree_numeric_features_v2 = [
    # Weather (raw 6h - 2h trends)
    "DryBulbTemperature_trend_2h_6h",
    "WindSpeed_trend_2h_6h",
    "RelativeHumidity_trend_2h_6h",
    "Visibility_trend_2h_6h",
    "PrecipBinary",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (short-term airport congestion)
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
    # Tail-number delay propagation
    "prev_DEP_DEL15", "turnaround_minutes",
]

tree_assembler_inputs_v2 = tree_numeric_features_v2 + [f"{c}_ohe" for c in categorical_features_v2]
tree_assembler_v2 = VectorAssembler(inputCols=tree_assembler_inputs_v2, outputCol="features", handleInvalid="skip")
tree_pipeline_stages_v2 = indexers_v2 + encoders_v2 + [tree_assembler_v2]

# 3. MLP Pipeline (reduced feature set for neural network)
#    Drops ORIGIN OHE (high cardinality), uses PageRank instead
mlp_numeric_features_v2 = [
    # Weather (standardized 6h - 2h trends)
    "DryBulbTemperature_trend_2h_6h_stdized",
    "WindSpeed_trend_2h_6h_stdized",
    "RelativeHumidity_trend_2h_6h_stdized",
    "Visibility_trend_2h_6h_stdized",
    "PrecipBinary",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (log of total flights replaces raw count)
    "delay_pct_flights_2_4hr_before", "log_total_flights_2_4hr_before",
    # Graph (replaces ORIGIN/DEST)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
    # Tail-number delay propagation
    "prev_DEP_DEL15", "turnaround_minutes",
]

mlp_categorical_features_v2 = [
    "OP_UNIQUE_CARRIER", "DAY_OF_WEEK", "MONTH", "DEP_HOUR",
]

mlp_indexers_v2 = [
    StringIndexer(inputCol=c, outputCol=f"{c}_mlp_idx", handleInvalid="keep")
    for c in mlp_categorical_features_v2
]
mlp_encoders_v2 = [
    OneHotEncoder(inputCol=f"{c}_mlp_idx", outputCol=f"{c}_mlp_ohe")
    for c in mlp_categorical_features_v2
]
mlp_assembler_inputs_v2 = mlp_numeric_features_v2 + [f"{c}_mlp_ohe" for c in mlp_categorical_features_v2]
mlp_assembler_v2 = VectorAssembler(inputCols=mlp_assembler_inputs_v2, outputCol="features", handleInvalid="skip")
mlp_pipeline_stages_v2 = mlp_indexers_v2 + mlp_encoders_v2 + [mlp_assembler_v2]

# Determine MLP input dimension from training data
mlp_prep_v2 = Pipeline(stages=mlp_pipeline_stages_v2).fit(otpw_train_df_v2)
sample_vec_v2 = mlp_prep_v2.transform(otpw_train_df_v2.limit(1)).select("features").head()[0]
mlp_input_dim_v2 = len(sample_vec_v2)

print(f"LR:    {len(lr_numeric_features_v2)} numeric + {len(categorical_features_v2)} categorical = {len(lr_assembler_inputs_v2)} assembler inputs")
print(f"Tree:  {len(tree_numeric_features_v2)} numeric + {len(categorical_features_v2)} categorical = {len(tree_assembler_inputs_v2)} assembler inputs")
print(f"MLP:   {len(mlp_numeric_features_v2)} numeric + {len(mlp_categorical_features_v2)} categorical = {mlp_input_dim_v2} input dim (after OHE)")


### Cross-Validation on the Extended Feature Set (v2)

Before the blind test we run the same three rolling folds used in v1 (Train 2015 -> Val 2016, Train 2016 -> Val 2017, Train 2017 -> Val 2018) on the v2 feature set. We only re-run the four models that selected from v1 (LR, GBT, MLP Deep, LSTM); GBT reuses the v1 grid-search winner so any v1-vs-v2 difference reflects features alone, not hyperparameters.


In [0]:
lr_cv_v2 = load_checkpoint("cv_v2_logistic_regression")
if lr_cv_v2 is None:
    lr_cv_v2, lr_cv_v2_models = rolling_cv(
        otpw_train_df_v2,
        model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
        pipeline_stages=lr_pipeline_stages_v2, folds=cv_folds
    )
    save_checkpoint(lr_cv_v2, "cv_v2_logistic_regression")
else:
    lr_cv_v2_models = []

display(lr_cv_v2.round(4))


In [0]:
# Reuse the v1 GBT grid-search winner so v1-vs-v2 reflects features, not hyperparameters
gbt_best_params_cv = gs_df.iloc[0]
gbt_maxIter_v2_cv = int(gbt_best_params_cv["maxIter"])
gbt_maxDepth_v2_cv = int(gbt_best_params_cv["maxDepth"])
gbt_stepSize_v2_cv = float(gbt_best_params_cv["stepSize"])

gbt_cv_v2 = load_checkpoint("cv_v2_gradient_boosted_trees")
if gbt_cv_v2 is None:
    gbt_cv_v2, gbt_cv_v2_models = rolling_cv(
        otpw_train_df_v2,
        model_fn=lambda: GBTClassifier(
            featuresCol="features", labelCol=label_col,
            maxIter=gbt_maxIter_v2_cv, maxDepth=gbt_maxDepth_v2_cv,
            stepSize=gbt_stepSize_v2_cv, seed=42
        ),
        pipeline_stages=tree_pipeline_stages_v2, folds=cv_folds
    )
    save_checkpoint(gbt_cv_v2, "cv_v2_gradient_boosted_trees")
else:
    gbt_cv_v2_models = []

print(f"GBT v2 CV hyperparameters (from v1 grid search winner): "
      f"maxIter={gbt_maxIter_v2_cv}, maxDepth={gbt_maxDepth_v2_cv}, stepSize={gbt_stepSize_v2_cv}")
display(gbt_cv_v2.round(4))


In [0]:
mlp_deep_cv_v2 = load_checkpoint("cv_v2_mlp_deep")
if mlp_deep_cv_v2 is None:
    fold_results = []
    mlp_deep_cv_v2_models = []

    for i, fold in enumerate(cv_folds, 1):
        train_fold = otpw_train_df_v2.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df_v2.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        # Compute feature dimension for this fold
        prep_pipeline = Pipeline(stages=mlp_pipeline_stages_v2).fit(train_balanced)
        sample_vec = prep_pipeline.transform(train_balanced).select("features").first()[0]
        fold_input_dim = len(sample_vec)

        # Deep MLP: 2 hidden layers [input, 128, 64, 2]
        mlp_model = MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[fold_input_dim, 128, 64, 2], maxIter=100, blockSize=128, seed=42
        )
        full_pipeline = Pipeline(stages=mlp_pipeline_stages_v2 + [mlp_model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        mlp_deep_cv_v2_models.append(fitted)

        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    mlp_deep_cv_v2 = pd.DataFrame(fold_results)
    save_checkpoint(mlp_deep_cv_v2, "cv_v2_mlp_deep")
else:
    mlp_deep_cv_v2_models = []

display(mlp_deep_cv_v2.round(4))


In [0]:
# LSTM CV on v2 feature set (default LSTM hyperparameters)
lstm_input_features_v2 = mlp_numeric_features_v2  # same numeric set as MLP v2
lstm_columns_to_collect_v2 = ["TAIL_NUM", "sched_depart_date_time", "YEAR"] + lstm_input_features_v2 + ["DEP_DEL15"]

lstm_cv_v2 = load_checkpoint("cv_v2_lstm")
if lstm_cv_v2 is None:
    fold_results = []

    for i, fold in enumerate(cv_folds, 1):
        train_fold = otpw_train_df_v2.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df_v2.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)
        train_size = train_balanced.count()

        train_pdf = train_balanced.select(lstm_columns_to_collect_v2).toPandas()
        test_pdf = test_fold.select(lstm_columns_to_collect_v2).toPandas()
        train_pdf = train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
        test_pdf = test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

        print(f"\nFold {i}: {fold['label']}")
        print(f"  Building sequences (train={len(train_pdf)}, test={len(test_pdf)})...")
        X_tr, y_tr = build_sequences(train_pdf, SEQ_LEN, lstm_input_features_v2, "DEP_DEL15")
        X_te, y_te = build_sequences(test_pdf, SEQ_LEN, lstm_input_features_v2, "DEP_DEL15")

        lstm_model, device, train_time = train_lstm(
            X_tr, y_tr, input_dim=len(lstm_input_features_v2), epochs=LSTM_EPOCHS
        )

        train_probs = predict_lstm(lstm_model, X_tr, device)
        test_probs = predict_lstm(lstm_model, X_te, device)
        train_preds = (train_probs >= 0.50).astype(float)
        test_preds = (test_probs >= 0.50).astype(float)

        train_metrics = evaluate_numpy(y_tr, train_preds, train_probs)
        test_metrics = evaluate_numpy(y_te, test_preds, test_probs)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_size, "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"  Training time: {train_time}s")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    lstm_cv_v2 = pd.DataFrame(fold_results)
    save_checkpoint(lstm_cv_v2, "cv_v2_lstm")

display(lstm_cv_v2.round(4))


In [0]:
# Compare v1 vs v2 CV mean Test F2 across the four models
cv_compare_models = ["Logistic Regression", "Gradient Boosted Trees", "MLP Deep", "LSTM"]
cv_v1_dfs = {"Logistic Regression": lr_cv, "Gradient Boosted Trees": gbt_cv, "MLP Deep": mlp_deep_cv, "LSTM": lstm_cv}
cv_v2_dfs = {"Logistic Regression": lr_cv_v2, "Gradient Boosted Trees": gbt_cv_v2, "MLP Deep": mlp_deep_cv_v2, "LSTM": lstm_cv_v2}

v1_means = [float(cv_v1_dfs[m]["Test_F2"].mean()) for m in cv_compare_models]
v2_means = [float(cv_v2_dfs[m]["Test_F2"].mean()) for m in cv_compare_models]
deltas = [v2 - v1 for v1, v2 in zip(v1_means, v2_means)]

cv_compare_df = pd.DataFrame({
    "Model": cv_compare_models,
    "v1_Mean_Test_F2": v1_means,
    "v2_Mean_Test_F2": v2_means,
    "Delta": deltas,
})
print("Cross-validation mean Test F2 (3 folds): v1 vs v2")
display(cv_compare_df.round(4))

x = np.arange(len(cv_compare_models))
width = 0.38
fig, ax = plt.subplots(1, 1, figsize=(10, 5.5))
b1 = ax.bar(x - width/2, v1_means, width, label="v1 (baseline features)", color="#4472C4", edgecolor="white")
b2 = ax.bar(x + width/2, v2_means, width, label="v2 (+ trends + tail-number)", color="#2E7D32", edgecolor="white")

for bars, vals in [(b1, v1_means), (b2, v2_means)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

for xi, d in zip(x, deltas):
    color = "#2E7D32" if d > 0.001 else "#C62828" if d < -0.001 else "#9E9E9E"
    ax.text(xi, max(v1_means[xi], v2_means[xi]) + 0.025, f"{d:+.4f}", ha="center", fontsize=9, color=color, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(cv_compare_models, fontsize=10)
ax.set_ylabel("Mean Test F2 (3 CV folds)", fontsize=11)
ax.set_title("v1 vs v2 Cross-Validation: Mean Test F2 across 2015->16, 2016->17, 2017->18", fontsize=12)
ax.legend(loc="lower right")
ax.set_ylim(0, max(max(v1_means), max(v2_means)) * 1.18)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


In [0]:
import os
_stale_v2_path = f"/dbfs/student-groups/Group_3_2/{CHECKPOINT_DATE}_v2_final_results_2019.parquet"
_existing_v2 = load_checkpoint("v2_final_results_2019")
if _existing_v2 is not None:
    _stale_models = set(_existing_v2["Model"]) - {"Logistic Regression", "Gradient Boosted Trees", "MLP Deep", "LSTM"}
    if _stale_models:
        print(f"Removing stale v2 checkpoint (contains dropped models: {sorted(_stale_models)})")
        dbutils.fs.rm(f"dbfs:/student-groups/Group_3_2/{CHECKPOINT_DATE}_v2_final_results_2019.parquet", recurse=True)
    else:
        print("v2 checkpoint already matches trimmed model set; nothing to clean.")
else:
    print("No existing v2 checkpoint; nothing to clean.")

### Blind Test Evaluation of All Models on the Extended Feature Set


In [0]:
final_results_v2_df = load_checkpoint("v2_final_results_2019")
final_models_v2 = {}

# Reuse the v1 GBT grid-search winner so the v2 GBT bar reflects the contribution
# of the new features, not a regression to default hyperparameters. Defined outside
# the if-block so the waterfall cell can refit the GBT on cached runs if needed.
gbt_best_params = gs_df.iloc[0]
gbt_maxIter_v2 = int(gbt_best_params["maxIter"])
gbt_maxDepth_v2 = int(gbt_best_params["maxDepth"])
gbt_stepSize_v2 = float(gbt_best_params["stepSize"])

if final_results_v2_df is None:
    train_balanced_full_v2 = undersample_majority(otpw_train_df_v2, label_col=label_col, seed=42)
    final_results_v2 = []

    # Logistic Regression
    lr_result_v2, lr_final_v2 = run_experiment(
        LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
        lr_pipeline_stages_v2, train_balanced_full_v2, otpw_test_df_v2,
        "Logistic Regression", "v2: Train 2015-2018, Test 2019"
    )
    final_results_v2.append(lr_result_v2)
    final_models_v2["lr"] = lr_final_v2

    # Gradient Boosted Trees (using the v1 grid-search winner)
    gbt_result_v2, gbt_final_v2 = run_experiment(
        GBTClassifier(
            featuresCol="features", labelCol=label_col,
            maxIter=gbt_maxIter_v2, maxDepth=gbt_maxDepth_v2,
            stepSize=gbt_stepSize_v2, seed=42
        ),
        tree_pipeline_stages_v2, train_balanced_full_v2, otpw_test_df_v2,
        "Gradient Boosted Trees", "v2: Train 2015-2018, Test 2019"
    )
    final_results_v2.append(gbt_result_v2)
    final_models_v2["gbt"] = gbt_final_v2

    # MLP Deep
    mlp_deep_result_v2, mlp_deep_final_v2 = run_experiment(
        MultilayerPerceptronClassifier(
            featuresCol="features", labelCol=label_col,
            layers=[mlp_input_dim_v2, 128, 64, 2], maxIter=100, blockSize=128, seed=42
        ),
        mlp_pipeline_stages_v2, train_balanced_full_v2, otpw_test_df_v2,
        "MLP Deep", "v2: Train 2015-2018, Test 2019"
    )
    final_results_v2.append(mlp_deep_result_v2)
    final_models_v2["mlp_deep"] = mlp_deep_final_v2

    # Re-threshold the GBT to t=0.35 once and persist it to the checkpoint so the
    # waterfall cell can read it back without refitting on cached runs.
    gbt_test_preds = gbt_final_v2.transform(otpw_test_df_v2)
    gbt_test_preds = gbt_test_preds.withColumn("prob_delayed", get_prob_delayed(col("probability")))
    gbt_test_preds_t35 = gbt_test_preds.withColumn(
        "prediction", when(col("prob_delayed") >= 0.35, 1.0).otherwise(0.0)
    )
    gbt_result_v2["Test_F2_t35"] = evaluate_model(gbt_test_preds_t35)["F2"]

    final_results_v2_df = pd.DataFrame(final_results_v2)
    save_checkpoint(final_results_v2_df, "v2_final_results_2019")

print(f"GBT v2 hyperparameters (from v1 grid search winner): "
      f"maxIter={gbt_maxIter_v2}, maxDepth={gbt_maxDepth_v2}, stepSize={gbt_stepSize_v2}")
display(final_results_v2_df.round(4))


In [0]:
# LSTM v2 blind test (default LSTM hyperparameters, new feature set)
lstm_input_features_v2 = mlp_numeric_features_v2  # same numeric set as MLP v2
lstm_columns_to_collect_v2 = ["TAIL_NUM", "sched_depart_date_time", "YEAR"] + lstm_input_features_v2 + ["DEP_DEL15"]

lstm_final_v2_df = load_checkpoint("v2_lstm_final_2019")
if lstm_final_v2_df is None:
    train_balanced_full_v2 = undersample_majority(otpw_train_df_v2, label_col=label_col, seed=42)

    train_pdf = train_balanced_full_v2.select(lstm_columns_to_collect_v2).toPandas()
    test_pdf = otpw_test_df_v2.select(lstm_columns_to_collect_v2).toPandas()
    train_pdf = train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
    test_pdf = test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

    print("Building train sequences (v2)...")
    X_train_v2, y_train_v2 = build_sequences(train_pdf, SEQ_LEN, lstm_input_features_v2, "DEP_DEL15")
    print(f"Train sequences: {X_train_v2.shape}")
    print("Building test sequences (v2)...")
    X_test_v2, y_test_v2 = build_sequences(test_pdf, SEQ_LEN, lstm_input_features_v2, "DEP_DEL15")
    print(f"Test sequences: {X_test_v2.shape}")

    lstm_model_v2, device, lstm_train_time_v2 = train_lstm(
        X_train_v2, y_train_v2, input_dim=len(lstm_input_features_v2), epochs=LSTM_EPOCHS
    )

    train_probs = predict_lstm(lstm_model_v2, X_train_v2, device)
    test_probs = predict_lstm(lstm_model_v2, X_test_v2, device)
    train_preds = (train_probs >= 0.50).astype(float)
    test_preds = (test_probs >= 0.50).astype(float)

    train_metrics = evaluate_numpy(y_train_v2, train_preds, train_probs)
    test_metrics = evaluate_numpy(y_test_v2, test_preds, test_probs)

    print(f"\nLSTM v2 (default HP) on Train 2015-2018, Test 2019")
    print(f"Training time: {lstm_train_time_v2}s")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"------------------------------------------")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    lstm_result_v2 = {
        "Model": "LSTM",
        "Dataset": "v2: Train 2015-2018, Test 2019",
        "Train Time (s)": lstm_train_time_v2,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }
    lstm_final_v2_df = pd.DataFrame([lstm_result_v2])
    save_checkpoint(lstm_final_v2_df, "v2_lstm_final_2019")
else:
    lstm_result_v2 = lstm_final_v2_df.iloc[0].to_dict()

# Append to combined v2 results
final_results_v2_df = pd.concat([final_results_v2_df, pd.DataFrame([lstm_result_v2])], ignore_index=True).drop_duplicates(subset=["Model"], keep="last").reset_index(drop=True)
display(lstm_final_v2_df.round(4))


### Feature-Set Impact: Baseline vs Extended

The grouped bar chart below quantifies the per-model effect of adding the weather-trend and tail-number features. Each pair of bars compares the model's blind-test F2 (at the default decision threshold of 0.50) under the two feature sets, with the marginal contribution annotated above the pair.


In [0]:
# Combine v1 and v2 results into a single comparison table/chart
_v1_main = load_checkpoint("cv3_final_results_2019")
_v1_lstm = load_checkpoint("cv3_lstm_final_2019")
v1_all = pd.concat([_v1_main, _v1_lstm], ignore_index=True) if (_v1_main is not None and _v1_lstm is not None) else (_v1_main if _v1_main is not None else final_results_df)

model_keys = ["Logistic Regression", "Gradient Boosted Trees", "MLP Deep", "LSTM"]
model_labels = ["LR", "GBT", "MLP\n(2 layers)", "LSTM"]

comparison_rows = []
for name in model_keys:
    v1_row = v1_all[v1_all["Model"] == name]
    v2_row = final_results_v2_df[final_results_v2_df["Model"] == name]
    comparison_rows.append({
        "Model": name,
        "v1_Train_F2": float(v1_row["Train_F2"].values[0]) if len(v1_row) else np.nan,
        "v1_Test_F2": float(v1_row["Test_F2"].values[0]) if len(v1_row) else np.nan,
        "v2_Train_F2": float(v2_row["Train_F2"].values[0]) if len(v2_row) else np.nan,
        "v2_Test_F2": float(v2_row["Test_F2"].values[0]) if len(v2_row) else np.nan,
        "v1_Test_Recall": float(v1_row["Test_Recall"].values[0]) if len(v1_row) else np.nan,
        "v2_Test_Recall": float(v2_row["Test_Recall"].values[0]) if len(v2_row) else np.nan,
        "v1_Test_Precision": float(v1_row["Test_Precision"].values[0]) if len(v1_row) else np.nan,
        "v2_Test_Precision": float(v2_row["Test_Precision"].values[0]) if len(v2_row) else np.nan,
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df["Test_F2_Delta"] = (comparison_df["v2_Test_F2"] - comparison_df["v1_Test_F2"]).round(4)
print("v1 vs v2 Test F2 (default threshold = 0.50):")
display(comparison_df[["Model", "v1_Test_F2", "v2_Test_F2", "Test_F2_Delta",
                        "v1_Test_Recall", "v2_Test_Recall",
                        "v1_Test_Precision", "v2_Test_Precision"]].round(4))

# Grouped bar chart
fig, ax = plt.subplots(1, 1, figsize=(11, 6))
x = np.arange(len(model_labels))
width = 0.35
v1_vals = comparison_df["v1_Test_F2"].fillna(0).values
v2_vals = comparison_df["v2_Test_F2"].fillna(0).values

ax.bar(x - width/2, v1_vals, width, label="v1 (Old features)", color="#9E9E9E")
ax.bar(x + width/2, v2_vals, width, label="v2 (+ Weather trend, Tail #)", color="#2E7D32")

for i in range(len(model_labels)):
    delta = v2_vals[i] - v1_vals[i]
    y_top = max(v1_vals[i], v2_vals[i])
    color = "#2E7D32" if delta > 0.001 else "#C62828" if delta < -0.001 else "gray"
    ax.text(x[i], y_top + 0.01, f"{delta:+.3f}", ha="center", fontsize=9, color=color, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=10)
ax.set_ylabel("Test F2 (2019 Blind Test, t=0.50)", fontsize=11)
ax.set_title("Feature-Set Impact: v1 (Old) vs v2 (Weather Trend + Tail # Propagation)", fontsize=13)
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, max(v2_vals.max(), v1_vals.max()) * 1.2)
plt.tight_layout()
plt.show()


### V2 Results Discussion

The V2 extended feature set produced the largest single improvement in the entire project. At the default threshold (t=0.50), GBT jumped from F2 = 0.50 (V1) to **F2 = 0.62** (+0.12), and at the operational threshold (t=0.35), GBT achieves **F2 = 0.63**. This improvement came from two new feature families:

- **Weather trend features** (6h-to-2h change in temperature, wind, humidity, visibility) capture whether conditions are worsening. It is a stronger predictor of delay than the static 2-hour snapshot alone.
- **Tail-number delay propagation** (`prev_DEP_DEL15`, `turnaround_minutes`) captures the cascading effect: if the same aircraft's previous leg was delayed, the next leg is more likely delayed.

Precision also improved dramatically, where it improved from 30% (V1) to 51% (V2) at t=0.50, confirming that the V1 gap analysis was correct: missing propagation features were a primary bottleneck. The V2 feature set directly addressed this.

LSTM also improving from F2 = 0.54 (V1) to F2 = 0.61 (V2) at t=0.50, though GBT is still the stronger model. To explore whether LSTM can close the gap, we next perform LSTM hyperparameter tuning on the V2 feature set, followed by a V3 feature experiment that adds absolute 2-hour weather levels back alongside the trends.


## LSTM Hyperparameter Tuning

In the baseline comparison the LSTM achieved competitive recall under default hyperparameters, so we will do a hyper tuning to determine whether it is really better than threshold-tuned GBT.


In [0]:
# Helper: train + evaluate a single LSTM config on a pre-built (X_tr, y_tr, X_te, y_te) split
def evaluate_lstm_config(X_tr, y_tr, X_te, y_te, input_dim,
                          hidden_dim=64, num_layers=2, dropout=0.3,
                          lr=0.001, epochs=10, batch_size=4096):
    model, device, train_time = train_lstm(
        X_tr, y_tr, input_dim=input_dim,
        epochs=epochs, hidden_dim=hidden_dim, num_layers=num_layers,
        dropout=dropout, lr=lr, batch_size=batch_size,
    )
    train_probs = predict_lstm(model, X_tr, device)
    test_probs = predict_lstm(model, X_te, device)

    train_preds_50 = (train_probs >= 0.50).astype(float)
    test_preds_50 = (test_probs >= 0.50).astype(float)
    test_preds_35 = (test_probs >= 0.35).astype(float)

    train_metrics = evaluate_numpy(y_tr, train_preds_50, train_probs)
    test_metrics_50 = evaluate_numpy(y_te, test_preds_50, test_probs)
    test_metrics_35 = evaluate_numpy(y_te, test_preds_35, test_probs)

    return {
        "Train_F2": train_metrics["F2"],
        "Test_F2_t50": test_metrics_50["F2"],
        "Test_F2_t35": test_metrics_35["F2"],
        "Test_Recall_t35": test_metrics_35["Recall"],
        "Test_Precision_t35": test_metrics_35["Precision"],
        "Test_AUC-PR": test_metrics_50["AUC-PR"],
        "Overfit_Gap": round(train_metrics["F2"] - test_metrics_50["F2"], 4),
        "Train_Time_s": train_time,
    }, model

print("Pre-building LSTM sequences (seq_len=5) ...")
_train_balanced_full_v2 = undersample_majority(otpw_train_df_v2, label_col=label_col, seed=42)
_train_pdf = _train_balanced_full_v2.select(lstm_columns_to_collect_v2).toPandas()
_test_pdf = otpw_test_df_v2.select(lstm_columns_to_collect_v2).toPandas()
_train_pdf = _train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
_test_pdf = _test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

X_tr_s5, y_tr_s5 = build_sequences(_train_pdf, 5, lstm_input_features_v2, "DEP_DEL15")
X_te_s5, y_te_s5 = build_sequences(_test_pdf, 5, lstm_input_features_v2, "DEP_DEL15")
print(f"Train: {X_tr_s5.shape}, Test: {X_te_s5.shape}")


In [0]:
lstm_gs_df = load_checkpoint("grid_search_lstm")
if lstm_gs_df is None:
    param_grid = [
        {"hidden_dim": h, "num_layers": n, "dropout": d}
        for h in [32, 64, 128]
        for n in [1, 2]
        for d in [0.2, 0.4]
    ]

    lstm_gs_results = []
    best_threshold = 0.35

    for idx, params in enumerate(param_grid, 1):
        metrics, _ = evaluate_lstm_config(
            X_tr_s5, y_tr_s5, X_te_s5, y_te_s5,
            input_dim=len(lstm_input_features_v2),
            hidden_dim=params["hidden_dim"], num_layers=params["num_layers"],
            dropout=params["dropout"], lr=0.001, epochs=10,
        )

        lstm_gs_results.append({
            "hidden_dim": params["hidden_dim"],
            "num_layers": params["num_layers"],
            "dropout": params["dropout"],
            "seq_len": 5, "lr": 0.001, "epochs": 10,
            **metrics,
        })

        print(
            f"[{idx}/{len(param_grid)}] hidden_dim={params['hidden_dim']:>3}, num_layers={params['num_layers']}, dropout={params['dropout']} | "
            f"Train F2={metrics['Train_F2']:.4f}, Test F2(t=0.50)={metrics['Test_F2_t50']:.4f}, "
            f"Test F2(t=0.35)={metrics['Test_F2_t35']:.4f} | {metrics['Train_Time_s']}s"
        )

    lstm_gs_df = pd.DataFrame(lstm_gs_results).sort_values("Test_F2_t35", ascending=False).reset_index(drop=True)
    save_checkpoint(lstm_gs_df, "grid_search_lstm")

print("\nLSTM Grid Search Results (sorted by Test F2 at t=0.35):")
display(lstm_gs_df.round(4))


In [0]:
# LSTM grid search summary
best_lstm = lstm_gs_df.iloc[0]
lstm_baseline_f2 = float(lstm_final_v2_df["Test_F2"].values[0])

print("LSTM Grid Search - Best configuration:")
print(f"  hidden_dim={int(best_lstm['hidden_dim'])}, num_layers={int(best_lstm['num_layers'])}, dropout={best_lstm['dropout']}")
print(f"  Train F2:        {best_lstm['Train_F2']:.4f}")
print(f"  Test F2 (t=0.50): {best_lstm['Test_F2_t50']:.4f}")
print(f"  Test F2 (t=0.35): {best_lstm['Test_F2_t35']:.4f}")
print(f"  Overfit Gap:     {best_lstm['Overfit_Gap']:+.4f}")
print(f"  AUC-PR:          {best_lstm['Test_AUC-PR']:.4f}")
print(f"  vs default LSTM: F2 change at t=0.50: {best_lstm['Test_F2_t50'] - lstm_baseline_f2:+.4f}")


### LSTM Tuning Outcome

The chart below traces the LSTM through three stages: the baseline configuration on the original feature set, the same configuration retrained on the extended feature set, and the best tuned configuration. Reporting both the default decision threshold (t=0.50) and the operationally tuned threshold (t=0.35).


In [0]:
_v1_lstm_test_f2 = float(_v1_lstm["Test_F2"].values[0]) if _v1_lstm is not None and len(_v1_lstm) else float(lstm_final_df["Test_F2"].values[0])
_v2_lstm_default_f2 = float(lstm_final_v2_df["Test_F2"].values[0])
_v2_lstm_tuned_f2_t50 = float(lstm_gs_df.iloc[0]["Test_F2_t50"])
_v2_lstm_tuned_f2_t35 = float(lstm_gs_df.iloc[0]["Test_F2_t35"])

stages = ["v1 LSTM\n(default HP)", "v2 LSTM\n(default HP, +new feats)", "v2 LSTM Tuned\n(t=0.50)", "v2 LSTM Tuned\n(t=0.35)"]
values = [_v1_lstm_test_f2, _v2_lstm_default_f2, _v2_lstm_tuned_f2_t50, _v2_lstm_tuned_f2_t35]
colors_l = ["#9E9E9E", "#1976D2", "#388E3C", "#2E7D32"]

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(range(len(stages)), values, color=colors_l, edgecolor="white", linewidth=1.5)
for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")
    if i > 0:
        delta = val - values[i-1]
        delta_color = "#2E7D32" if delta > 0.001 else "#C62828" if delta < -0.001 else "gray"
        ax.text(bar.get_x() + bar.get_width()/2, val - 0.025, f"{delta:+.4f}", ha="center", fontsize=9, color=delta_color)

ax.set_xticks(range(len(stages)))
ax.set_xticklabels(stages, fontsize=9)
ax.set_ylabel("Test F2 (2019 Blind Test)", fontsize=11)
ax.set_title("LSTM Progression: New Features + Hyperparameter Tuning", fontsize=13)
ax.set_ylim(0, max(values) * 1.25)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"\nLSTM total improvement: {values[-1] - values[0]:+.4f} F2 (v1 default {values[0]:.4f} -> v2 tuned t=0.35 {values[-1]:.4f})")


# V3 Feature Set: Trend + 2h Features Combined

The v2 feature set replaced the static 2-hour weather snapshot with weather trends (6h - 2h change in temperature, wind, humidity, visibility). V3 brings the absolute `2h_Hourly*` weather levels back and adds `VisibilityCat`, keeping them alongside the v2 trend features so the model can see both the level and the direction of change.

Because v2 already identified GBT and LSTM as the two strongest models, v3 re-evaluates only those two. No hyperparameter tuning is performed in v3: GBT reuses the v1 grid-search winner and LSTM reuses the v2 grid-search winner. The goal is to isolate the contribution of the additional `2h_Hourly*` + `VisibilityCat` features. Undersampling stays at 1:1.


In [0]:
otpw_train_df_v3 = otpw_train_df_v2
otpw_test_df_v3 = otpw_test_df_v2

print(f"v3 train rows: {otpw_train_df_v3.count():,}")
print(f"v3 test rows:  {otpw_test_df_v3.count():,}")


In [0]:
# Categoricals add VisibilityCat on top of v2
categorical_features_v3 = [
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR", "VisibilityCat",
]

indexers_v3 = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features_v3
]
encoders_v3 = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features_v3
]

# Tree/GBT pipeline: raw v2 trends + absolute 2h_Hourly* levels
tree_numeric_features_v3 = [
    # Weather trends (v2)
    "DryBulbTemperature_trend_2h_6h",
    "WindSpeed_trend_2h_6h",
    "RelativeHumidity_trend_2h_6h",
    "Visibility_trend_2h_6h",
    # Absolute 2h_Hourly* levels (new in v3)
    "2h_HourlyDryBulbTemperature",
    "2h_HourlyWindSpeed",
    "2h_HourlyRelativeHumidity",
    "PrecipBinary",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (short-term airport congestion)
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
    # Tail-number delay propagation
    "prev_DEP_DEL15", "turnaround_minutes",
]

tree_assembler_inputs_v3 = tree_numeric_features_v3 + [f"{c}_ohe" for c in categorical_features_v3]
tree_assembler_v3 = VectorAssembler(inputCols=tree_assembler_inputs_v3, outputCol="features", handleInvalid="skip")
tree_pipeline_stages_v3 = indexers_v3 + encoders_v3 + [tree_assembler_v3]

# LSTM inputs: standardized versions of v2 trends + standardized absolute 2h_Hourly* levels
lstm_input_features_v3 = [
    "DryBulbTemperature_trend_2h_6h_stdized",
    "WindSpeed_trend_2h_6h_stdized",
    "RelativeHumidity_trend_2h_6h_stdized",
    "Visibility_trend_2h_6h_stdized",
    "2h_HourlyDryBulbTemperature_stdized",
    "2h_HourlyWindSpeed_stdized",
    "2h_HourlyRelativeHumidity_stdized",
    "PrecipBinary",
    "DISTANCE", "IsWeekend",
    "delay_pct_flights_2_4hr_before", "log_total_flights_2_4hr_before",
    "origin_pagerank_weighted", "origin_pagerank_unweighted",
    "dest_pagerank_weighted", "dest_pagerank_unweighted",
    "prev_DEP_DEL15", "turnaround_minutes",
]
lstm_columns_to_collect_v3 = ["TAIL_NUM", "sched_depart_date_time", "YEAR"] + lstm_input_features_v3 + ["DEP_DEL15"]

print(f"Tree/GBT v3: {len(tree_numeric_features_v3)} numeric + {len(categorical_features_v3)} categorical = {len(tree_assembler_inputs_v3)} assembler inputs")
print(f"LSTM v3:     {len(lstm_input_features_v3)} numeric inputs")


## V3 Cross-Validation (GBT + LSTM Only)

Three rolling folds (2015->16, 2016->17, 2017->18), same as v1 and v2.


In [0]:
# GBT v3 CV: reuses the v1 grid-search winner hyperparameters
gbt_maxIter_v3 = int(gs_df.iloc[0]["maxIter"])
gbt_maxDepth_v3 = int(gs_df.iloc[0]["maxDepth"])
gbt_stepSize_v3 = float(gs_df.iloc[0]["stepSize"])

gbt_cv_v3 = load_checkpoint("cv_v3_gradient_boosted_trees")
if gbt_cv_v3 is None:
    gbt_cv_v3, gbt_cv_v3_models = rolling_cv(
        otpw_train_df_v3,
        model_fn=lambda: GBTClassifier(
            featuresCol="features", labelCol=label_col,
            maxIter=gbt_maxIter_v3, maxDepth=gbt_maxDepth_v3,
            stepSize=gbt_stepSize_v3, seed=42
        ),
        pipeline_stages=tree_pipeline_stages_v3, folds=cv_folds
    )
    save_checkpoint(gbt_cv_v3, "cv_v3_gradient_boosted_trees")
else:
    gbt_cv_v3_models = []

print(f"GBT v3 CV hyperparameters: maxIter={gbt_maxIter_v3}, maxDepth={gbt_maxDepth_v3}, stepSize={gbt_stepSize_v3}")
display(gbt_cv_v3.round(4))


In [0]:
# LSTM v3 CV: reuses the v2 grid-search winner hyperparameters
lstm_hidden_dim_v3 = int(lstm_gs_df.iloc[0]["hidden_dim"])
lstm_num_layers_v3 = int(lstm_gs_df.iloc[0]["num_layers"])
lstm_dropout_v3 = float(lstm_gs_df.iloc[0]["dropout"])

lstm_cv_v3 = load_checkpoint("cv_v3_lstm")
if lstm_cv_v3 is None:
    fold_results = []

    for i, fold in enumerate(cv_folds, 1):
        train_fold = otpw_train_df_v3.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = otpw_train_df_v3.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)
        train_size = train_balanced.count()

        train_pdf = train_balanced.select(lstm_columns_to_collect_v3).toPandas()
        test_pdf = test_fold.select(lstm_columns_to_collect_v3).toPandas()
        train_pdf = train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
        test_pdf = test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

        print(f"\nFold {i}: {fold['label']}")
        print(f"  Building sequences (train={len(train_pdf)}, test={len(test_pdf)})...")
        X_tr, y_tr = build_sequences(train_pdf, SEQ_LEN, lstm_input_features_v3, "DEP_DEL15")
        X_te, y_te = build_sequences(test_pdf, SEQ_LEN, lstm_input_features_v3, "DEP_DEL15")

        lstm_model, device, train_time = train_lstm(
            X_tr, y_tr,
            input_dim=len(lstm_input_features_v3),
            hidden_dim=lstm_hidden_dim_v3,
            num_layers=lstm_num_layers_v3,
            dropout=lstm_dropout_v3,
            epochs=LSTM_EPOCHS,
        )

        train_probs = predict_lstm(lstm_model, X_tr, device)
        test_probs = predict_lstm(lstm_model, X_te, device)
        train_preds = (train_probs >= 0.50).astype(float)
        test_preds = (test_probs >= 0.50).astype(float)

        train_metrics = evaluate_numpy(y_tr, train_preds, train_probs)
        test_metrics = evaluate_numpy(y_te, test_preds, test_probs)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_size, "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"  Training time: {train_time}s")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    lstm_cv_v3 = pd.DataFrame(fold_results)
    save_checkpoint(lstm_cv_v3, "cv_v3_lstm")

print(f"\nLSTM v3 CV hyperparameters (from v2 grid search winner): "
      f"hidden_dim={lstm_hidden_dim_v3}, num_layers={lstm_num_layers_v3}, dropout={lstm_dropout_v3}")
display(lstm_cv_v3.round(4))


## V3 Cross-Validation Results Discussion

The V3 feature set adds absolute 2-hour weather observations (`2h_HourlyDryBulbTemperature`, `2h_HourlyWindSpeed`, `2h_HourlyRelativeHumidity`) and `VisibilityCat` back alongside the V2 trend features. We want to see if models benefit from seeing both the weather level and the direction of change.

Cross-validation across three rolling folds (2015 to 16, 2016 to 17, 2017 to 18) shows that V3 maintains almost the same performance to V2. GBT achieves average CV Test F2 in the 0.60 - 0.62, consistent with V2 CV results. This confirms that may provide marginal improvement.

The blind test on the held-out 2019 data was not completed at submission time. The blind test code is included at the end of this notebook; if execution completes, results will appear in the output cells.

## Experimentation Progression

The waterfall chart below traces F2-score changes across the entire project, from the Phase II raw baseline through V2 feature engineering and LSTM hyperparameter tuning. Each bar shows the cumulative F2 at that step, with the marginal delta annotated. The V3 cross-validation results are discussed above; the V3 blind test is included as an appendix at the end of this notebook.


In [0]:
# Pull V2 GBT t=0.35 for the waterfall
_gbt_v2 = final_results_v2_df[final_results_v2_df["Model"] == "Gradient Boosted Trees"]
_gbt_v2_test_f2_t35 = np.nan
if "Test_F2_t35" in _gbt_v2.columns and len(_gbt_v2) and not pd.isna(_gbt_v2["Test_F2_t35"].values[0]):
    _gbt_v2_test_f2_t35 = float(_gbt_v2["Test_F2_t35"].values[0])

# Best LSTM tuned (t=0.35) from grid search
_best_lstm_f2_t35 = float(lstm_gs_df.iloc[0]["Test_F2_t35"])

# Build the waterfall (8 bars: Phase II baseline -> V2 LSTM tuned t=0.35)
steps = [
    "Phase II\nRaw Baseline",
    "+ Basic FE",
    "+ Time-Series\n(RFM)",
    "Phase III v1\n4yr + New FE",
    "+ Threshold\n(t=0.35)",
    "+ HP Tuning\n(GBT)",
    "+ V2 Features\n(Trend + Tail #)\nGBT t=0.35",
    "+ LSTM Tuned\n(t=0.35)",
]
f2_values = [
    0.4868, 0.4875, 0.4674, 0.5047, 0.5598, 0.5629,
    _gbt_v2_test_f2_t35,
    _best_lstm_f2_t35,
]
deltas = [0] + [f2_values[i] - f2_values[i-1] for i in range(1, len(f2_values))]
colors = ["#4472C4"] + [
    "#2E7D32" if d > 0.001 else "#C62828" if d < -0.001 else "#9E9E9E"
    for d in deltas[1:]
]

fig, ax = plt.subplots(1, 1, figsize=(13, 6))
bars = ax.bar(range(len(steps)), f2_values, color=colors, edgecolor="white", linewidth=1.5)
for i, (bar, val, delta) in enumerate(zip(bars, f2_values, deltas)):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.4f}",
                ha="center", fontsize=9, fontweight="bold")
    if i > 0 and not np.isnan(delta):
        delta_color = "#2E7D32" if delta > 0.001 else "#C62828" if delta < -0.001 else "#9E9E9E"
        ax.text(bar.get_x() + bar.get_width()/2, val - 0.025, f"{delta:+.4f}",
                ha="center", fontsize=8, color=delta_color)

ax.axvline(x=2.5, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=5.5, color="gray", linestyle="--", alpha=0.5)
ax.text(1.0, 0.68, "Phase II (2015)", ha="center", fontsize=9, color="gray", style="italic")
ax.text(4.0, 0.68, "Phase III v1 (2015-2019)", ha="center", fontsize=9, color="gray", style="italic")
ax.text(6.5, 0.68, "Phase III v2", ha="center", fontsize=9, color="gray", style="italic")

ax.set_xticks(range(len(steps)))
ax.set_xticklabels(steps, fontsize=9)
ax.set_ylabel("F2 Score (2019 Blind Test)", fontsize=11)
ax.set_title("Experimentation Progression: Phase II Baseline → V2 Features + LSTM Tuning", fontsize=13)
ax.set_ylim(0.40, 0.72)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"\nTotal F2 improvement (Phase II Baseline → V2 LSTM tuned t=0.35): "
      f"{f2_values[-1] - f2_values[0]:+.4f}")
print(f"V2 features contribution (vs V1 HP-tuned GBT t=0.35): "
      f"{f2_values[6] - f2_values[5]:+.4f}")
print(f"LSTM tuning contribution (vs V2 GBT t=0.35): "
      f"{f2_values[7] - f2_values[6]:+.4f}")


# Conclusion

This project is trying to predict whether a U.S. domestic flight would be delayed by 15 or more minutes, two hours before departure, to enable Airline XYZ's Operations Control Center to take proactive action. We framed this as a binary classification problem, prioritizing F2-score to minimize missed delays while maintaining actionable predictions.

## Best Model

**Gradient Boosted Trees with the V2 extended feature set** is our best completed model. At the default threshold (t=0.50), it achieves **F2 = 0.6198** on the 2019 blind test with **65% recall** and **51% precision**. At the operational threshold (t=0.35), F2 rises to **0.6336**. Compared to the V1 baseline GBT (F2 = 0.50), the V2 feature set alone contributed a +0.12 F2 improvement, the largest gain in the project so far.

The V2 feature set adds two families on top of the V1 baseline:
- **Weather trend features** (6h-to-2h change in temperature, wind, humidity, visibility): captures whether conditions are worsening, not just the current snapshot
- **Tail-number delay propagation** (`prev_DEP_DEL15`, `turnaround_minutes`): captures whether the same aircraft's previous leg was delayed and how much turnaround time is available

## What Drove Performance

| Factor | F2 Contribution |
|--------|----------------|
| Scaling to 4 years + graph/congestion features (V1) | Baseline (0.497) |
| Threshold optimization (0.50 → 0.35) | +0.062 |
| Hyperparameter tuning (GBT grid search) | +0.003 |
| **V2 feature engineering (trends + tail #)** | **+0.057 (t=0.50) / +0.071 (t=0.35)** |

Feature engineering and operational threshold design accounted for over **95% of all improvement**. Hyperparameter tuning was near-irrelevant. This confirms that in applied ML, the data you build matters more than the model you choose.

## Operational Recommendation

Deploy V2 GBT at threshold 0.35. For every flight flagged as high-risk, the OCC should pre-position standby crew, reserve backup gates, and prepare passenger rebooking options. At t=0.50, the model's 51% precision means roughly half of flagged flights are actual delays, which leads a significant improvement over the V1 model's 30% precision, making the alerts more actionable and reducing false alert.

## Future Work

1. **V3 blind test validation:** The V3 feature set (adding absolute 2-hour weather levels alongside V2 trends) was validated through 3-fold cross-validation and shows comparable performance. The blind test on the 2019 held-out data was still executing at submission time due to the computational cost of training GBT (100 trees, depth 8) and LSTM on the full 4-year dataset. The code is included in the appendix section below and will produce results if execution completes. However, we are not expecting significant improvement from V2 result since from the cross validation result comparison, we noticed very minimal improvement.

2. **Combine multiple models together:** The two models make different kinds of mistakes, GBT works on each flight in isolation while LSTM looks back at the same aircraft's recent flights, so modeling based on combination of their predictions should catch some delays that either one alone would miss. A simple regresison model, take into the prediction result of both models might boost the result even further.

3. **Undersampling ratio sweep:** All current results use a 1:1 majority-to-minority undersample. The 50/50 training distribution is far from the natural ~80/20, which inflates false positives at test time. Sweeping less aggressive ratios (e.g., 1.5:1, 2:1, 3:1) trades some recall for precision and may produce a higher F2 at the operational threshold.

4. **Error analysis on misclassified test flights:** Pull the 2019 blind-test predictions where the best model is wrong and segment them by carrier, origin/destination, hour, day of week, weather regime, and tail-number history. Patterns in the misclassifications will reveal the next feature or model change worth attempting.


In [0]:
print("MLP Deep:", load_checkpoint("cv3_mlp_deep")[["Test_F2","Test_Recall","Test_Precision","Train_Time_s"]].mean().round(4).to_dict())
print("LSTM v1:", load_checkpoint("cv3_lstm_final_2019")[["Test_F2","Test_Recall","Test_Precision","Train Time (s)"]].iloc[0].round(4).to_dict())
print("LSTM v2:", load_checkpoint("v2_lstm_final_2019")[["Test_F2","Test_Recall","Test_Precision","Train Time (s)"]].iloc[0].round(4).to_dict())
print("LSTM grid best:", load_checkpoint("grid_search_lstm").iloc[0][["Test_F2_t50","Test_F2_t35","Test_Recall_t35","Test_Precision_t35","Train_Time_s"]].round(4).to_dict())

In [0]:
print(load_checkpoint("grid_search_lstm")[["hidden_dim","num_layers","dropout","Train_Time_s"]].round(1).to_string())

# Appendix: V3 Blind Test

The V3 feature set was validated through 3-fold rolling cross-validation above, confirming that adding absolute 2-hour weather levels alongside the trend features maintains comparable F2 performance to V2. The blind test on the 2019 data is included in the cells below. This execution was still in progress when we submit. If execution completes, the results will appear in the output cells below.


In [0]:
# GBT v3 blind test on 2019 (same HP as CV)
final_results_v3_df = load_checkpoint("v3_final_results_2019")
final_models_v3 = {}

if final_results_v3_df is None:
    train_balanced_full_v3 = undersample_majority(otpw_train_df_v3, label_col=label_col, seed=42)

    gbt_result_v3, gbt_final_v3 = run_experiment(
        GBTClassifier(
            featuresCol="features", labelCol=label_col,
            maxIter=gbt_maxIter_v3, maxDepth=gbt_maxDepth_v3,
            stepSize=gbt_stepSize_v3, seed=42
        ),
        tree_pipeline_stages_v3, train_balanced_full_v3, otpw_test_df_v3,
        "Gradient Boosted Trees", "v3: Train 2015-2018, Test 2019"
    )
    final_models_v3["gbt"] = gbt_final_v3

    # Persist the GBT t=0.35 score alongside the default-threshold row so the
    # waterfall cell can read it back without refitting on cached runs.
    gbt_test_preds = gbt_final_v3.transform(otpw_test_df_v3)
    gbt_test_preds = gbt_test_preds.withColumn("prob_delayed", get_prob_delayed(col("probability")))
    gbt_test_preds_t35 = gbt_test_preds.withColumn(
        "prediction", when(col("prob_delayed") >= 0.35, 1.0).otherwise(0.0)
    )
    gbt_result_v3["Test_F2_t35"] = evaluate_model(gbt_test_preds_t35)["F2"]

    final_results_v3_df = pd.DataFrame([gbt_result_v3])
    save_checkpoint(final_results_v3_df, "v3_final_results_2019")

print(f"GBT v3 hyperparameters: maxIter={gbt_maxIter_v3}, maxDepth={gbt_maxDepth_v3}, stepSize={gbt_stepSize_v3}")
display(final_results_v3_df.round(4))


In [0]:
# GBT v3 Train 
final_results_v3_df = load_checkpoint("v3_final_results_2019")
final_models_v3 = {}

if final_results_v3_df is None:
    train_balanced_full_v3 = undersample_majority(otpw_train_df_v3, label_col=label_col, seed=42).cache()
    print(f"Balanced train rows: {train_balanced_full_v3.count():,}")

    otpw_test_df_v3 = otpw_test_df_v3.cache()
    print(f"Test rows: {otpw_test_df_v3.count():,}")

    # Train
    gbt_pipeline_v3 = Pipeline(stages=tree_pipeline_stages_v3 + [
        GBTClassifier(
            featuresCol="features", labelCol=label_col,
            maxIter=gbt_maxIter_v3, maxDepth=gbt_maxDepth_v3,
            stepSize=gbt_stepSize_v3, seed=42
        )
    ])
    start = time.time()
    gbt_final_v3 = gbt_pipeline_v3.fit(train_balanced_full_v3)
    gbt_train_time_v3 = round(time.time() - start, 1)
    final_models_v3["gbt"] = gbt_final_v3
    print(f"GBT v3 training done: {gbt_train_time_v3}s")
else:
    print("Checkpoint loaded — skipping GBT v3 training.")
    gbt_final_v3 = None

print(f"GBT v3 HP: maxIter={gbt_maxIter_v3}, maxDepth={gbt_maxDepth_v3}, stepSize={gbt_stepSize_v3}")


In [0]:
# GBT v3 Step t=0.50
if final_results_v3_df is None and gbt_final_v3 is not None:
    gbt_train_preds_v3 = gbt_final_v3.transform(train_balanced_full_v3).cache()
    gbt_test_preds_v3 = gbt_final_v3.transform(otpw_test_df_v3).cache()
    gbt_train_preds_v3.count()
    gbt_test_preds_v3.count()
    print("Predictions cached.")

    train_metrics = evaluate_model(gbt_train_preds_v3)
    test_metrics = evaluate_model(gbt_test_preds_v3)

    print(f"\nGBT v3 on v3: Train 2015-2018, Test 2019")
    print(f"Training time: {gbt_train_time_v3}s")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"------------------------------------------")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    gbt_result_v3 = {
        "Model": "Gradient Boosted Trees",
        "Dataset": "v3: Train 2015-2018, Test 2019",
        "Train Time (s)": gbt_train_time_v3,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }
else:
    print("Skipped — checkpoint already loaded or model not available.")


In [0]:
# GBT v3 Threshold t=0.35 + Checkpoint ──
if final_results_v3_df is None and gbt_final_v3 is not None:
    gbt_test_preds_v3 = gbt_test_preds_v3.withColumn(
        "prob_delayed", get_prob_delayed(col("probability"))
    )
    gbt_test_preds_t35 = gbt_test_preds_v3.withColumn(
        "prediction", when(col("prob_delayed") >= 0.35, 1.0).otherwise(0.0)
    ).cache()
    gbt_test_preds_t35.count()

    gbt_result_v3["Test_F2_t35"] = evaluate_model(gbt_test_preds_t35)["F2"]
    print(f"GBT v3 Test F2 (t=0.35): {gbt_result_v3['Test_F2_t35']:.4f}")

    final_results_v3_df = pd.DataFrame([gbt_result_v3])
    save_checkpoint(final_results_v3_df, "v3_final_results_2019")

    gbt_train_preds_v3.unpersist()
    gbt_test_preds_v3.unpersist()
    gbt_test_preds_t35.unpersist()
    train_balanced_full_v3.unpersist()
    print("Cached DataFrames unpersisted.")
else:
    print("Skipped — checkpoint already loaded or model not available.")

display(final_results_v3_df.round(4))


In [0]:
# LSTM v3 Step 1/3: Prepare data + Build sequences
lstm_final_v3_df = load_checkpoint("v3_lstm_final_2019")

if lstm_final_v3_df is None:
    train_balanced_full_v3 = undersample_majority(otpw_train_df_v3, label_col=label_col, seed=42).cache()
    train_balanced_full_v3.count()

    print("Collecting train data to pandas...")
    train_pdf = train_balanced_full_v3.select(lstm_columns_to_collect_v3).toPandas()
    print("Collecting test data to pandas...")
    test_pdf = otpw_test_df_v3.select(lstm_columns_to_collect_v3).toPandas()

    train_pdf = train_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)
    test_pdf = test_pdf.sort_values(["TAIL_NUM", "sched_depart_date_time"]).reset_index(drop=True)

    print("Building train sequences (v3)...")
    X_train_v3, y_train_v3 = build_sequences(train_pdf, SEQ_LEN, lstm_input_features_v3, "DEP_DEL15")
    print(f"Train sequences: {X_train_v3.shape}")
    print("Building test sequences (v3)...")
    X_test_v3, y_test_v3 = build_sequences(test_pdf, SEQ_LEN, lstm_input_features_v3, "DEP_DEL15")
    print(f"Test sequences: {X_test_v3.shape}")

    train_balanced_full_v3.unpersist()
    print("Data prepared. Ready to train LSTM.")
else:
    print("Checkpoint loaded — skipping LSTM v3 data prep.")


In [0]:
# LSTM v3 Step 2/3: Train + Predict
if lstm_final_v3_df is None:
    lstm_model_v3, device, lstm_train_time_v3 = train_lstm(
        X_train_v3, y_train_v3,
        input_dim=len(lstm_input_features_v3),
        hidden_dim=lstm_hidden_dim_v3,
        num_layers=lstm_num_layers_v3,
        dropout=lstm_dropout_v3,
        epochs=LSTM_EPOCHS,
    )
    print(f"LSTM v3 training done: {lstm_train_time_v3}s")

    train_probs = predict_lstm(lstm_model_v3, X_train_v3, device)
    test_probs = predict_lstm(lstm_model_v3, X_test_v3, device)
    print("Predictions complete.")
else:
    print("Skipped - checkpoint already loaded.")


In [0]:
# LSTM v3 Step 3/3: Evaluate + Checkpoint ───────
if lstm_final_v3_df is None:
    train_preds_t50 = (train_probs >= 0.50).astype(float)
    test_preds_t50 = (test_probs >= 0.50).astype(float)
    test_preds_t35 = (test_probs >= 0.35).astype(float)

    train_metrics_t50 = evaluate_numpy(y_train_v3, train_preds_t50, train_probs)
    test_metrics_t50 = evaluate_numpy(y_test_v3, test_preds_t50, test_probs)
    test_metrics_t35 = evaluate_numpy(y_test_v3, test_preds_t35, test_probs)

    print(f"LSTM v3 (v2 tuned HP) on Train 2015-2018, Test 2019")
    print(f"Training time: {lstm_train_time_v3}s")
    print(f"{'Metric':<20} {'Train':>10} {'Test t=0.50':>12} {'Test t=0.35':>12}")
    print(f"----------------------------------------------------------")
    for m in train_metrics_t50:
        print(f"{m:<20} {train_metrics_t50[m]:>10.4f} {test_metrics_t50[m]:>12.4f} {test_metrics_t35[m]:>12.4f}")

    lstm_result_v3 = {
        "Model": "LSTM",
        "Dataset": "v3: Train 2015-2018, Test 2019",
        "Train Time (s)": lstm_train_time_v3,
        **{f"Train_{k}": v for k, v in train_metrics_t50.items()},
        **{f"Test_{k}": v for k, v in test_metrics_t50.items()},
        "Test_F2_t35": test_metrics_t35["F2"],
        "Test_Recall_t35": test_metrics_t35["Recall"],
        "Test_Precision_t35": test_metrics_t35["Precision"],
    }
    lstm_final_v3_df = pd.DataFrame([lstm_result_v3])
    save_checkpoint(lstm_final_v3_df, "v3_lstm_final_2019")
else:
    lstm_result_v3 = lstm_final_v3_df.iloc[0].to_dict()

# Append to combined v3 results
final_results_v3_df = pd.concat(
    [final_results_v3_df, pd.DataFrame([lstm_result_v3])], ignore_index=True
).drop_duplicates(subset=["Model"], keep="last").reset_index(drop=True)
display(lstm_final_v3_df.round(4))


In [0]:
# v1 vs v2 vs v3 Test F2 for GBT and LSTM
_v3_models = ["Gradient Boosted Trees", "LSTM"]
_v3_labels = ["GBT", "LSTM"]

def _get_f2(df, model_name, col_name):
    row = df[df["Model"] == model_name]
    if len(row) == 0 or col_name not in row.columns:
        return np.nan
    v = row[col_name].values[0]
    return float(v) if not pd.isna(v) else np.nan

_v1_all = v1_all  # reused from the v1-vs-v2 chart cell

v3_compare_rows = []
for name in _v3_models:
    v3_compare_rows.append({
        "Model": name,
        "v1_Test_F2":  _get_f2(_v1_all, name, "Test_F2"),
        "v2_Test_F2":  _get_f2(final_results_v2_df, name, "Test_F2"),
        "v3_Test_F2":  _get_f2(final_results_v3_df, name, "Test_F2"),
    })
v3_compare_df = pd.DataFrame(v3_compare_rows)
v3_compare_df["v2_vs_v1"] = (v3_compare_df["v2_Test_F2"] - v3_compare_df["v1_Test_F2"]).round(4)
v3_compare_df["v3_vs_v2"] = (v3_compare_df["v3_Test_F2"] - v3_compare_df["v2_Test_F2"]).round(4)
print("v1 vs v2 vs v3 Test F2 (t=0.50), GBT and LSTM:")
display(v3_compare_df.round(4))

x = np.arange(len(_v3_labels))
width = 0.26
fig, ax = plt.subplots(1, 1, figsize=(9, 5.5))
b1 = ax.bar(x - width, v3_compare_df["v1_Test_F2"].fillna(0).values, width, label="v1 (old features)", color="#9E9E9E", edgecolor="white")
b2 = ax.bar(x,         v3_compare_df["v2_Test_F2"].fillna(0).values, width, label="v2 (+ trend + tail #)", color="#1976D2", edgecolor="white")
b3 = ax.bar(x + width, v3_compare_df["v3_Test_F2"].fillna(0).values, width, label="v3 (+ 2h_Hourly + VisibilityCat)", color="#2E7D32", edgecolor="white")

for bars, vals in [(b1, v3_compare_df["v1_Test_F2"]), (b2, v3_compare_df["v2_Test_F2"]), (b3, v3_compare_df["v3_Test_F2"])]:
    for bar, v in zip(bars, vals):
        if not pd.isna(v):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

for xi, d in zip(x, v3_compare_df["v3_vs_v2"]):
    if pd.isna(d):
        continue
    color = "#2E7D32" if d > 0.001 else "#C62828" if d < -0.001 else "#9E9E9E"
    y_top = max(v3_compare_df.iloc[xi][["v1_Test_F2", "v2_Test_F2", "v3_Test_F2"]].max(), 0)
    ax.text(xi + width, y_top + 0.025, f"v3-v2: {d:+.4f}", ha="center", fontsize=9, color=color, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(_v3_labels, fontsize=10)
ax.set_ylabel("Test F2 (2019 Blind Test, t=0.50)", fontsize=11)
ax.set_title("Feature-Set Impact: v1 vs v2 vs v3 (GBT and LSTM)", fontsize=12)
ax.legend(loc="lower right", fontsize=9)
ax.set_ylim(0, max(v3_compare_df[["v1_Test_F2", "v2_Test_F2", "v3_Test_F2"]].max().max(), 0.01) * 1.25)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


# Hardcoded Reproduction of Key Charts

The two cells below regenerate the experimentation-progression waterfall and the cross-model blind-test comparison from hardcoded numbers, so the charts can be reproduced for the submission notebook without re-running the full Spark / PyTorch pipeline.

In [0]:
import matplotlib.pyplot as plt
import numpy as np

steps_hc = [
    "Phase II\nRaw Baseline",
    "+ Basic FE",
    "+ Time-Series\n(RFM)",
    "Phase III v1\n4yr + New FE",
    "+ Threshold\n(t=0.35)",
    "+ HP Tuning\n(GBT)",
    "+ V2 Features\n(Trend + Tail #)\nGBT t=0.35",
    "+ LSTM Tuned\n(t=0.35)",
]
f2_values_hc = [0.4868, 0.4875, 0.4674, 0.5047, 0.5598, 0.5629, 0.6336, 0.6118]
deltas_hc = [0] + [f2_values_hc[i] - f2_values_hc[i-1] for i in range(1, len(f2_values_hc))]
colors_hc = ["#4472C4"] + [
    "#2E7D32" if d > 0.001 else "#C62828" if d < -0.001 else "#9E9E9E"
    for d in deltas_hc[1:]
]

fig, ax = plt.subplots(1, 1, figsize=(13, 6))
bars = ax.bar(range(len(steps_hc)), f2_values_hc, color=colors_hc,
              edgecolor="white", linewidth=1.5)
for i, (bar, val, delta) in enumerate(zip(bars, f2_values_hc, deltas_hc)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.4f}",
            ha="center", fontsize=9, fontweight="bold")
    if i > 0:
        delta_color = "#2E7D32" if delta > 0.001 else "#C62828" if delta < -0.001 else "#9E9E9E"
        ax.text(bar.get_x() + bar.get_width()/2, val - 0.025, f"{delta:+.4f}",
                ha="center", fontsize=8, color=delta_color)

ax.axvline(x=2.5, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=5.5, color="gray", linestyle="--", alpha=0.5)
ax.text(1.0, 0.68, "Phase II (2015)", ha="center", fontsize=9, color="gray", style="italic")
ax.text(4.0, 0.68, "Phase III v1 (2015-2019)", ha="center", fontsize=9, color="gray", style="italic")
ax.text(6.5, 0.68, "Phase III v2", ha="center", fontsize=9, color="gray", style="italic")

ax.set_xticks(range(len(steps_hc)))
ax.set_xticklabels(steps_hc, fontsize=9)
ax.set_ylabel("F2 Score (2019 Blind Test)", fontsize=11)
ax.set_title("Experimentation Progression: Phase II Baseline to Phase III V1 Feature to V2 Features + LSTM Tuning", fontsize=13)
ax.set_ylim(0.40, 0.72)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"\nTotal F2 improvement (Phase II Baseline -> V2 LSTM tuned t=0.35): "
      f"{f2_values_hc[-1] - f2_values_hc[0]:+.4f}")
print(f"V2 features contribution (vs V1 HP-tuned GBT t=0.35): "
      f"{f2_values_hc[6] - f2_values_hc[5]:+.4f}")
print(f"LSTM tuning contribution (vs V2 GBT t=0.35): "
      f"{f2_values_hc[7] - f2_values_hc[6]:+.4f}")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

model_names_hc = ["LR", "RF", "GBT", "MLP\n(1 layer)", "MLP\n(2 layers)", "LSTM"]
train_f2_hc = [0.5180, 0.6420, 0.5510, 0.4820, 0.5290, 0.7350]
test_f2_hc  = [0.5050, 0.4895, 0.4974, 0.4699, 0.5029, 0.5391]

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
x = np.arange(len(model_names_hc))
width = 0.3
ax.bar(x - width/2, train_f2_hc, width, label="Train F2", color="steelblue")
ax.bar(x + width/2, test_f2_hc, width, label="Test F2", color="darkorange")

for i in range(len(model_names_hc)):
    gap = train_f2_hc[i] - test_f2_hc[i]
    y_top = max(train_f2_hc[i], test_f2_hc[i])
    ax.text(x[i], y_top + 0.01, f"gap:{gap:+.3f}",
            ha="center", fontsize=8, color="gray")

ax.set_xticks(x)
ax.set_xticklabels(model_names_hc, fontsize=9)
ax.set_ylabel("F2 Score", fontsize=11)
ax.set_title("Train vs Test F2 - All Models on 2019 Blind Test (V1, t=0.50)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, 0.85)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---- Hardcoded per-fold V2 CV results --------------------------------------
cv_v2 = {
    "Logistic Regression": pd.DataFrame({
        "Fold":            [1, 2, 3],
        "Train_F2":        [0.6568, 0.6514, 0.6495],
        "Test_F2":         [0.5547, 0.5708, 0.5638],
        "Train_Recall":    [0.6353, 0.6292, 0.6258],
        "Test_Recall":     [0.6102, 0.6387, 0.6150],
        "Train_Precision": [0.7594, 0.7586, 0.7655],
        "Test_Precision":  [0.4066, 0.4006, 0.4230],
        "Train_AUC-PR":    [0.7962, 0.7970, 0.7992],
        "Test_AUC-PR":     [0.5064, 0.5231, 0.5158],
    }),
    "Gradient Boosted Trees": pd.DataFrame({
        "Fold":            [1, 2, 3],
        "Train_F2":        [0.6978, 0.6996, 0.7111],
        "Test_F2":         [0.6030, 0.6199, 0.6085],
        "Train_Recall":    [0.6720, 0.6740, 0.6872],
        "Test_Recall":     [0.6345, 0.6624, 0.6483],
        "Train_Precision": [0.8244, 0.8246, 0.8259],
        "Test_Precision":  [0.5032, 0.4934, 0.4884],
        "Train_AUC-PR":    [0.8577, 0.8603, 0.8640],
        "Test_AUC-PR":     [0.6220, 0.6376, 0.6270],
    }),
    "MLP Deep": pd.DataFrame({
        "Fold":            [1, 2, 3],
        "Train_F2":        [0.6074, 0.5906, 0.5847],
        "Test_F2":         [0.4495, 0.4575, 0.4468],
        "Train_Recall":    [0.6141, 0.5900, 0.5819],
        "Test_Recall":     [0.5944, 0.5862, 0.5504],
        "Train_Precision": [0.5822, 0.5928, 0.5962],
        "Test_Precision":  [0.2276, 0.2436, 0.2549],
        "Train_AUC-PR":    [0.6365, 0.6413, 0.6427],
        "Test_AUC-PR":     [0.3019, 0.3118, 0.3194],
    }),
    "LSTM": pd.DataFrame({
        "Fold":            [1, 2, 3],
        "Train_F2":        [0.6751, 0.6217, 0.6604],
        "Test_F2":         [0.5919, 0.5818, 0.5992],
        "Train_Recall":    [0.6477, 0.5819, 0.6282],
        "Test_Recall":     [0.6166, 0.5825, 0.6197],
        "Train_Precision": [0.8123, 0.8558, 0.8309],
        "Test_Precision":  [0.5100, 0.5790, 0.5290],
        "Train_AUC-PR":    [0.8433, 0.8426, 0.8468],
        "Test_AUC-PR":     [0.6215, 0.6367, 0.6355],
    }),
}

# ---- Per-fold consistency grid (4 metrics x 4 models, train vs validation) -
models = ["Logistic Regression", "Gradient Boosted Trees", "MLP Deep", "LSTM"]
metrics = ["F2", "Recall", "Precision", "AUC-PR"]
fold_labels = ["Fold 1\n(2015 > 16)", "Fold 2\n(2016 > 17)", "Fold 3\n(2017 > 18)"]

fig, axes = plt.subplots(len(metrics), len(models),
                         figsize=(5 * len(models), 4 * len(metrics)),
                         sharey="row")
x = np.arange(len(fold_labels))
width = 0.3

for row, metric in enumerate(metrics):
    for col_idx, model_name in enumerate(models):
        ax = axes[row, col_idx]
        df = cv_v2[model_name]
        train_col = f"Train_{metric}"
        test_col = f"Test_{metric}"

        ax.bar(x - width/2, df[train_col], width, label="Train", color="steelblue")
        ax.bar(x + width/2, df[test_col], width, label="Validation", color="darkorange")
        ax.set_xticks(x)
        ax.set_xticklabels(fold_labels, fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")

        for j in range(len(df)):
            gap = df[train_col].iloc[j] - df[test_col].iloc[j]
            y_top = max(df[train_col].iloc[j], df[test_col].iloc[j])
            ax.text(x[j], y_top + 0.01, f"{gap:+.2f}", ha="center", fontsize=7)

        if col_idx == 0:
            ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(model_name)
        if row == 0 and col_idx == len(models) - 1:
            ax.legend(fontsize=8)

fig.suptitle("V2 Cross-Validation: Per-Fold Train vs Validation (Rolling Year-Based Folds)",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()